In [1]:
# =============================================================================
# ===== CÉLULA 1 — SETUP, DADOS E REGIME DE SINAL =============================
# =============================================================================

import json, time, hashlib, warnings, platform, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import spearmanr, beta

# Avisos de convergência não são silenciados: um modelo que não convergiu é
# informação, não ruído. São contados e relatados.
warnings.simplefilter("always", ConvergenceWarning)
T_INICIO = time.time()

# -----------------------------------------------------------------------------
# Parâmetros
# -----------------------------------------------------------------------------
CFG = dict(
    SEMENTES             = (1807, 2024, 3141, 5926, 8979),
    N_FRAGMENTOS         = 5000,
    N_DOBRAS             = 5,
    FRACAO_PARADA        = 0.15,
    # Treinamento
    MAX_EPOCAS           = 80,
    MIN_EPOCAS           = 12,
    PACIENCIA            = 10,
    JANELA_SUAVIZACAO    = 5,
    TAXA_APRENDIZADO     = 3e-4,
    DECAIMENTO_PESO      = 1e-4,
    TAMANHO_LOTE         = 16,
    DIM_OCULTA           = 128,
    CLIP_GRADIENTE       = 5.0,
    # Triagem de desempenho mínimo, usada só na análise de sensibilidade.
    # Aplicada ao máximo da curva SUAVIZADA no conjunto de parada: o máximo
    # da curva bruta cruza qualquer limiar fixo com frequência apreciável,
    # porque são dezenas de avaliações ruidosas.
    LIMIAR_PARADA        = 0.60,
    # Pré-processamento
    PADRONIZA_BOLSA      = True,
    BOLSAS_NA_GPU        = True,
    N_COMPONENTES_OCUP   = 64,
    COMPRIMENTO_MIN_HIST = 60,
    COMPRIMENTO_MAX_HIST = 420,
    FRACAO_MAX_FORA_HIST = 0.01,
    # Estatística
    N_PERMUTACOES        = 200,
    N_BOOTSTRAP          = 2000,
    N_BOOTSTRAP_VARIANCIA= 400,
    K_COMPONENTES_PCA    = (1, 2, 5, 10, 25),
    ESPECIFICIDADE_ALVO  = 0.95,
    R2_ABORTA_RESIDUO    = 0.05,
    R2_INTERPRETA_RESIDUO= 0.30,
    # Diagnóstico
    N_PACIENTES_FORENSE  = 60,
    N_PACIENTES_FINITUDE = 30,
    N_PACIENTES_POR_CLASSE_D = 60,
    JANELA_EXTRA_ENCODER = 12.0,
    # --- NOVO NESTA REVISÃO ---------------------------------------------------
    # Um fragmento de fita dupla tem DOIS extremos 5', um em cada ponta. A
    # convenção do campo é contar os dois: o de montante lido para a frente,
    # o de jusante revertido e complementado.
    MOTIF_AMBOS_EXTREMOS = True,
    # Dividir a bolsa ao meio e comparar as metades responde se 5.000
    # fragmentos bastam para estimar um perfil de 256 categorias.
    MOTIF_TESTE_METADES  = True,
    # Metas de confiabilidade usadas para projetar quanta profundidade seria
    # necessária, por Spearman-Brown.
    MOTIF_METAS          = (0.70, 0.80, 0.90),
    # Retenção mínima de coorte aceitável ao propor uma bolsa maior. Perder
    # metade dos pacientes para ganhar precisão troca um problema por outro
    # pior, porque a perda não seria aleatória.
    RETENCAO_MINIMA      = 0.90,
    # Duas verificações de robustez que não produzem nenhum número citado no
    # artigo: a distribuição nula do vetor completo do Caduceus e a versão
    # padronizada da PCA da ocupação. Ligue para auditar; custam ~6 minutos.
    VERIFICACOES_EXTRA   = False,
)
SEMENTE_MESTRA = CFG["SEMENTES"][0]

HASH_CFG = hashlib.sha256(
    json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()[:16]

np.random.seed(SEMENTE_MESTRA)
torch.manual_seed(SEMENTE_MESTRA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ESTUDO_DEV, ESTUDO_EXT = "cristiano2019_delfi_wgs", "jiang2015_hcc_wgs"
SAUDAVEL, HCC = {"healthy"}, "liver_cancer"

print("=" * 100)
print("ONDE O SINAL SE PERDE ENTRE 5.000 FRAGMENTOS E UM DIAGNÓSTICO")
print("Auditoria de Multiple Instance Learning em fragmentômica de cfDNA — revisão 2")
print("=" * 100)
print(f"Assinatura da configuração : {HASH_CFG}")
print(f"Repetições independentes   : {len(CFG['SEMENTES'])} sementes")
print(f"Coorte externa             : exploratória, tarefa distinta")
print(f"Verificações extras        : {'ligadas' if CFG['VERIFICACOES_EXTRA'] else 'desligadas'}")
print(f"Hardware                   : {DISPOSITIVO}")

# -----------------------------------------------------------------------------
# Glossário dos braços. Nomes por extenso em toda a saída; as formas curtas
# só aparecem em eixos de gráfico, onde não cabe o nome inteiro.
#
# A cobertura é chamada de "DELFI-like" e não de "DELFI". Ela é restrita ao
# painel de promotores expandido e normalizada pela extensão acessível em cada
# janela; não é reprodução do método original sobre o genoma inteiro.
# -----------------------------------------------------------------------------
NOME = {
    "cad_lin":          "Caduceus (Média → Logística)",
    "cad_vec_lin":      "Caduceus + Comprimento (Média → Logística)",
    "cad_vec_lin_frag": "Caduceus + Comprimento (Média → Logística, escala de fragmento)",
    "cad_vec_mlpdir":   "Caduceus + Comprimento (Média → MLP)",
    "cad_vec_mlp":      "Caduceus + Comprimento (Média → Projeção → MLP)",
    "cad_mean":         "Caduceus + Comprimento (Projeção → Média → MLP)",
    "cad_att":          "Caduceus + Comprimento (Projeção → Atenção MIL)",
    "kmer_vec_mlp":     "4-mers + Comprimento (Média → Projeção → MLP)",
    "kmer_mean":        "4-mers + Comprimento (Projeção → Média → MLP)",
    "kmer_att":         "4-mers + Comprimento (Projeção → Atenção MIL)",
    "length":           "Somente Comprimento (Atenção MIL) [canal único]",
    "delfi":            "Cobertura DELFI-like, painel inteiro (MLP)",
    "delfi_prom_all":   "Cobertura DELFI-like, promotora, todos os fragmentos (MLP)",
    "delfi_prom_matched": "Cobertura DELFI-like, promotora, os mesmos 5.000 (MLP)",
    "rep_kmer":         "Composição de 4-mers (Média → Logística)",
    "rep_len":          "Histograma de Comprimentos (Logística)",
    "rep_delfi":        "Cobertura DELFI-like, painel inteiro (Logística)",
    "rep_delfi_prom_all": "Cobertura DELFI-like, promotora, todos os fragmentos (Logística)",
    "rep_delfi_prom_matched": "Cobertura DELFI-like, promotora, os mesmos 5.000 (Logística)",
    "rep_motif":        "Perfil de End-Motif (Logística)",
    "rep_cad_resid":    "Caduceus — Resíduo após 4-mers e Ocupação (Logística)",
}
CURTO = {
    "cad_lin": "Cad: Média→Log",              "cad_vec_lin": "Cad+Comp: Média→Log",
    "cad_vec_lin_frag": "Cad+Comp: Média→Log (frag)",
    "cad_vec_mlpdir": "Cad+Comp: Média→MLP",  "cad_vec_mlp": "Cad+Comp: Média→Proj→MLP",
    "cad_mean": "Cad+Comp: Proj→Média→MLP",   "cad_att": "Cad+Comp: Proj→Atenção",
    "kmer_vec_mlp": "4mer: Média→Proj→MLP",   "kmer_mean": "4mer: Proj→Média→MLP",
    "kmer_att": "4mer: Proj→Atenção",         "length": "Só Comprimento: Atenção",
    "delfi": "Cobertura painel: MLP",
    "delfi_prom_all": "Cobertura promotora: MLP",
    "delfi_prom_matched": "Cobertura promotora 5.000: MLP",
    "rep_kmer": "4mers: Log",                 "rep_len": "Comprimento: Log",
    "rep_delfi": "Cobertura painel: Log",
    "rep_delfi_prom_all": "Cobertura promotora: Log",
    "rep_delfi_prom_matched": "Cobertura promotora 5.000: Log",
    "rep_motif": "End-Motif: Log",
    "rep_cad_resid": "Cad Resíduo: Log",
}
def nome(k): return NOME.get(k, k)

print("\n" + "-" * 100)
print("AMBIENTE DE SOFTWARE")
print("-" * 100)
for lib, ver in [("python", platform.python_version()), ("numpy", np.__version__),
                 ("pandas", pd.__version__), ("scikit-learn", sklearn.__version__),
                 ("torch", torch.__version__), ("seaborn", sns.__version__)]:
    print(f"  {lib:<16s} {ver}")
if torch.cuda.is_available():
    print(f"  {'gpu':<16s} {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------------
# Localização dos arquivos e assinatura das entradas
# -----------------------------------------------------------------------------
def acha(padrao, obrigatorio=True):
    """Primeiro arquivo que casa com o padrão, em qualquer dataset anexado."""
    achados = sorted(Path("/kaggle/input").rglob(padrao))
    if obrigatorio and not achados:
        anexados = sorted(p.name for p in Path("/kaggle/input").iterdir() if p.is_dir())
        raise RuntimeError(
            f"Arquivo obrigatório não encontrado: {padrao}\n"
            f"Datasets anexados: {anexados}\n"
            f"Anexe cfdna-promoter-multimodal-resume-v3 e cfdna-ref-hg38.")
    return achados[0] if achados else None

PASTA = None
for c in Path("/kaggle/input").rglob("02_features"):
    if c.is_dir():
        PASTA = c; break
if PASTA is None:
    raise RuntimeError("Pasta '02_features' não encontrada. Anexe o dataset ao notebook.")
SAIDA = Path("/kaggle/working/04_results"); SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    "metadados":            "metadata.csv",
    "caduceus":             "caduceus_embeddings.npy",
    "delfi":                "delfi_all.npy",
    "delfi_prom_all":       "delfi_promoter_all.npy",
    "delfi_prom_matched":   "delfi_promoter_matched.npy",
    "kmers":                "kmer_features.npy",
    "comprimento":          "promoter_fragment_lengths.npy",
    "promotores":           "promoter_ids.npy",
    "coordenadas":          "promoter_fragment_coordinates.npy",
}
for _, arq in ARQUIVOS.items():
    if not (PASTA / arq).exists():
        raise RuntimeError(f"Arquivo obrigatório ausente em 02_features: {arq}")

print("\n" + "-" * 100)
print("ASSINATURA DOS ARQUIVOS DE ENTRADA (SHA-256)")
print("-" * 100)
print(f"  Entrada : {PASTA}")
print(f"  Saída   : {SAIDA}\n")
t0 = time.time(); hashes = {}
for _, arq in ARQUIVOS.items():
    h = hashlib.sha256()
    with open(PASTA / arq, "rb") as f:
        for bloco in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloco)
    hashes[arq] = h.hexdigest()
    print(f"  {arq:<38s} {h.hexdigest()[:24]}...")
HASH_DADOS = hashlib.sha256(json.dumps(hashes, sort_keys=True).encode()).hexdigest()[:16]
print(f"\n  Assinatura agregada : {HASH_DADOS}   ({time.time()-t0:.0f} s)")

# mmap_mode="r" mapeia o arquivo sem carregá-lo na RAM. As matrizes de embedding
# têm ~2 GB cada; lê-se só o pedaço necessário.
META  = pd.read_csv(PASTA / ARQUIVOS["metadados"])
CAD   = np.load(PASTA / ARQUIVOS["caduceus"], mmap_mode="r")
KMER  = np.load(PASTA / ARQUIVOS["kmers"],    mmap_mode="r")
COMP  = np.load(PASTA / ARQUIVOS["comprimento"])
PROM  = np.load(PASTA / ARQUIVOS["promotores"], mmap_mode="r")
DELFI = np.load(PASTA / ARQUIVOS["delfi"]).astype(np.float32)
DELFI_PROM_ALL     = np.load(PASTA / ARQUIVOS["delfi_prom_all"]).astype(np.float32)
DELFI_PROM_MATCHED = np.load(PASTA / ARQUIVOS["delfi_prom_matched"]).astype(np.float32)

N_PAC = len(META)
D_EMB = CAD.shape[2]

# -----------------------------------------------------------------------------
# GATE — consistência estrutural e finitude.
#
# Verifica que todos os arquivos têm o mesmo número de linhas, a mesma
# geometria, e que nenhum valor é infinito ou indefinido.
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("GATE — CONSISTÊNCIA ESTRUTURAL E FINITUDE")
print("=" * 100)
falhas = []
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER), ("Comprimentos", COMP),
                ("Promotores", PROM), ("Cobertura painel", DELFI),
                ("Cobertura promotora", DELFI_PROM_ALL),
                ("Cobertura promotora 5.000", DELFI_PROM_MATCHED)]:
    if arr.shape[0] != N_PAC:
        falhas.append(f"{nm}: {arr.shape[0]} linhas, metadados têm {N_PAC}")
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER), ("Comprimentos", COMP),
                ("Promotores", PROM)]:
    if arr.shape[1] != CFG["N_FRAGMENTOS"]:
        falhas.append(f"{nm}: {arr.shape[1]} fragmentos por paciente, "
                      f"esperado {CFG['N_FRAGMENTOS']}")
if KMER.shape[2] != D_EMB:
    falhas.append(f"4-mers têm {KMER.shape[2]} dimensões, Caduceus tem {D_EMB}")
# As duas coberturas promotoras precisam ter a MESMA geometria. É isso que torna
# o degrau PROFUNDIDADE uma mudança de uma coisa só.
if DELFI_PROM_ALL.shape[1] != DELFI_PROM_MATCHED.shape[1]:
    falhas.append(
        f"As duas coberturas promotoras têm dimensões diferentes "
        f"({DELFI_PROM_ALL.shape[1]} e {DELFI_PROM_MATCHED.shape[1]}). "
        f"O degrau PROFUNDIDADE mudaria mais de uma coisa.")
for nm, arr in [("Cobertura painel", DELFI), ("Cobertura promotora", DELFI_PROM_ALL),
                ("Cobertura promotora 5.000", DELFI_PROM_MATCHED),
                ("Comprimentos", COMP)]:
    if not np.isfinite(arr).all():
        falhas.append(f"{nm} contém valores não finitos")
# Nos embeddings a verificação é uma TRIAGEM amostral: varrer 4 GB duas vezes
# não muda nenhum número, e corrupção sistemática aparece na amostra.
_am_fin = np.sort(np.random.RandomState(SEMENTE_MESTRA).choice(
    N_PAC, min(CFG["N_PACIENTES_FINITUDE"], N_PAC), replace=False))
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER)]:
    if not np.isfinite(np.asarray(arr[_am_fin], dtype=np.float32)).all():
        falhas.append(f"{nm} contém valores não finitos (triagem de {len(_am_fin)} pacientes)")
if PROM.max() < 0 or PROM.min() < -1:
    falhas.append("Identificadores de promotor fora da faixa esperada")
if falhas:
    for f in falhas: print(f"  [FALHA] {f}")
    raise RuntimeError("Estrutura dos arquivos inconsistente.")

N_PROM = int(PROM.max()) + 1   # definido só depois de PROM passar no gate de faixa

print("  Aprovado: formas e tipos conferem.")
print(f"  Cobertura no painel inteiro     : {DELFI.shape[1]} colunas")
print(f"  Cobertura promotora             : {DELFI_PROM_ALL.shape[1]} colunas")
print(f"  Cobertura promotora, 5.000      : {DELFI_PROM_MATCHED.shape[1]} colunas")
print(f"  As duas promotoras compartilham a geometria — o degrau PROFUNDIDADE")
print(f"  altera apenas o número de moléculas contadas.")
print(f"  Finitude completa  : coberturas, comprimentos")
print(f"  Finitude amostral  : Caduceus, 4-mers ({len(_am_fin)} pacientes sorteados)")

# -----------------------------------------------------------------------------
# GATE — a cobertura restrita é realmente diferente da cobertura completa?
#
# Se os três arquivos fossem cópias, a escada da cobertura mediria zero por
# construção e ninguém perceberia. Basta um número para descartar essa hipótese.
# -----------------------------------------------------------------------------
dif_regiao = float(np.abs(DELFI_PROM_ALL - DELFI_PROM_MATCHED).mean())
if dif_regiao < 1e-8:
    raise RuntimeError(
        "As coberturas promotoras completa e restrita são idênticas. "
        "O degrau PROFUNDIDADE não teria o que medir.")
print(f"  Diferença média entre a cobertura promotora completa e a restrita "
      f"aos 5.000 : {dif_regiao:.4f}")

# -----------------------------------------------------------------------------
# GATE — as bolsas estão completas?
#
# Se um paciente tivesse menos de 5.000 fragmentos e o resto fosse preenchido
# com zeros, a padronização transformaria esses zeros em valores não nulos. A
# média e os pesos da atenção passariam a depender da quantidade de
# preenchimento, não da biologia.
# -----------------------------------------------------------------------------
n_zero = int((COMP == 0).sum())
print("\n" + "-" * 100)
print("GATE — BOLSAS COMPLETAS (AUSÊNCIA DE PREENCHIMENTO ARTIFICIAL)")
print("-" * 100)
print(f"  Posições com comprimento zero : {n_zero} de {COMP.size}")
if n_zero > 0:
    raise RuntimeError(
        f"{n_zero} posições vazias detectadas. Sem máscara explícita, médias e "
        f"pesos de atenção ficariam contaminados pela fração de preenchimento.")
print(f"  Aprovado: todos os {N_PAC} pacientes têm {CFG['N_FRAGMENTOS']} fragmentos reais.")

print("\n" + "-" * 100)
print("COMPOSIÇÃO DAS COORTES")
print("-" * 100)
print(META.groupby(["estudo", "condicao"]).size().to_string())

IDX_DEV = np.where((META["estudo"] == ESTUDO_DEV).values)[0]
IDX_EXT = np.where((META["estudo"] == ESTUDO_EXT).values)[0]

# -----------------------------------------------------------------------------
# ESCALA DOS CANAIS E QUANTO A BOLSA REALMENTE CARREGA
#
# (a) ESCALA. O comprimento de um fragmento varia de 60 a 420 pares de base.
#     Cada dimensão do Caduceus varia em torno de 0,004. Sem padronização, o
#     comprimento dominaria a rede por duas ordens de grandeza.
#
# (b) PONTO DE CROSSOVER DA VARIÂNCIA. Cada dimensão varia por dois motivos:
#     fragmentos do mesmo paciente diferem entre si (variância DENTRO, σ²d) e
#     pacientes diferem entre si (variância ENTRE, σ²e). A média de n
#     fragmentos tem variância
#
#          σ²e  +  σ²d / n
#
#     O segundo termo encolhe com n; o primeiro não. Onde os dois se igualam,
#     n* = σ²d/σ²e, o erro de estimar a média deixa de ser o termo dominante.
#     A σ²e observada já contém σ²d/5000, então a corrigimos. O cálculo não usa
#     rótulo de doença.
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("ESCALA DOS CANAIS E PONTO DE CROSSOVER DA VARIÂNCIA (só Baltimore)")
print("=" * 100)
amo = np.sort(np.random.RandomState(SEMENTE_MESTRA).choice(
    IDX_DEV, size=min(CFG["N_PACIENTES_FORENSE"], len(IDX_DEV)), replace=False))

def crossover_variancia(bolsa, n_boot, semente):
    """n* = σ²dentro / σ²entre, com intervalo por reamostragem de PACIENTES."""
    var_d_pac = bolsa.var(axis=1)
    medias    = bolsa.mean(axis=1)
    def calcula(idx):
        vd = var_d_pac[idx].mean(axis=0)
        vo = medias[idx].var(axis=0, ddof=1)
        ve = np.maximum(vo - vd / CFG["N_FRAGMENTOS"], 1e-24)
        return vd / ve
    n_est = calcula(np.arange(len(bolsa)))
    rng = np.random.RandomState(semente); n = len(bolsa)
    boots = [float(np.median(calcula(rng.randint(0, n, n)))) for _ in range(n_boot)]
    return (float(np.median(n_est)),
            float(np.percentile(n_est, 25)), float(np.percentile(n_est, 75)),
            float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)))

forense = {}
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER)]:
    b = np.asarray(arr[amo], dtype=np.float32)
    ne, q25, q75, blo, bhi = crossover_variancia(
        b, CFG["N_BOOTSTRAP_VARIANCIA"], SEMENTE_MESTRA)
    forense[nm] = dict(dp_frag=float(b.std(axis=1).mean()),
                       dp_medias=float(b.mean(axis=1).std(axis=0, ddof=1).mean()),
                       n_estrela=ne, iqr_dim=(q25, q75), ic_pacientes=(blo, bhi))
    f = forense[nm]
    print(f"\n  {nm}")
    print(f"    Desvio entre fragmentos do mesmo paciente (σd) : {f['dp_frag']:.5f}")
    print(f"    Desvio entre pacientes, observado (σe)         : {f['dp_medias']:.5f}")
    print(f"    Crossover n* = σ²d/σ²e (mediana das dimensões) : {ne:.0f}")
    print(f"      dispersão entre dimensões (quartis)          : {q25:.0f} – {q75:.0f}")
    print(f"      IC 95% por reamostragem de pacientes         : {blo:.0f} – {bhi:.0f}")
    del b

N_ESTRELA = forense["Caduceus"]["n_estrela"]
N_EST_LO, N_EST_HI = forense["Caduceus"]["ic_pacientes"]

print(f"\n  LEITURA: até cerca de {N_ESTRELA:.0f} fragmentos o erro de estimação da média")
print(f"  domina; a partir daí, domina a variação entre pacientes. Fragmentos de um")
print(f"  mesmo paciente compartilham região genômica, protocolo e composição, não")
print(f"  contam como observações independentes. É o efeito de desenho clássico de")
print(f"  amostragem por conglomerado.")

print(f"\n  Escala relativa na primeira camada:")
print(f"    Comprimento, depois de padronizado : desvio 1,000")
for nm in ["Caduceus", "4-mers"]:
    dp = forense[nm]["dp_frag"]
    print(f"    Cada dimensão de {nm:<10s}         : desvio {dp:.5f}  → {1.0/dp:.0f}x menor")
print(f"  Por isso a padronização de cada canal é obrigatória, e está ativa.")

# -----------------------------------------------------------------------------
# O que estes «4-mers» medem?
#
# Se cada fragmento contribuísse com UM 4-mer (o motivo de corte), cada dimensão
# seria um ensaio de Bernoulli com p = 1/256. Se contamos os 4-mers ao longo de
# toda a janela lida pelo encoder, a média de muitos ensaios reduz o desvio.
# Comparar as duas previsões com o observado diz qual das duas descreve o
# arquivo. Importa porque o achado central é «o encoder é composição de bases» —
# e o leitor precisa saber de que composição estamos falando.
# -----------------------------------------------------------------------------
comp_amo   = COMP[amo]
comp_med   = float(np.median(comp_amo[comp_amo > 0]))
janela     = comp_med + CFG["JANELA_EXTRA_ENCODER"]
dp_motivo  = float(np.sqrt((1/256.0) * (1 - 1/256.0)))
dp_composi = dp_motivo / np.sqrt(max(janela - 3.0, 1.0))
dp_obs     = forense["4-mers"]["dp_frag"]

print("\n" + "-" * 100)
print("O QUE OS 4-MERS MEDEM")
print("-" * 100)
print(f"  Janela lida pelo encoder : {janela:.0f} pb (fragmento mediano {comp_med:.0f} pb)")
print(f"  Desvio por dimensão      : motivo de corte {dp_motivo:.4f} · "
      f"composição da janela {dp_composi:.4f} · OBSERVADO {dp_obs:.5f}")
print(f"  O observado é {dp_obs/dp_composi:.2f}x a previsão de composição de janela e "
      f"{dp_obs/dp_motivo:.3f}x a de motivo.")
print(f"  Compatível, em ordem de grandeza, com composição da janela. As 4 bases do")
print(f"  ponto de corte pesam {100*4.0/janela:.1f}% da janela: o motivo terminal NÃO é")
print(f"  medido por esta representação. A Célula 3 o mede diretamente.")

# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 100)
print("  • Os arquivos de features não carregam identificador de paciente. Que a")
print("    linha 173 de cada arquivo seja o mesmo paciente é pressuposto da etapa")
print("    de geração; aqui só a dimensão é verificada.")
print("  • A finitude dos embeddings é uma triagem em "
      f"{len(_am_fin)} pacientes, não uma varredura.")
print("  • n* é um ponto de crossover sob o modelo de variância adotado, que supõe")
print("    correlação intra-bolsa homogênea. Não é tamanho amostral efetivo no")
print("    sentido clássico, e não prova que os demais fragmentos sejam inúteis.")
print("  • A amostra mistura casos e controles: a doença contribui para σ²e e")
print("    portanto reduz n*. O valor é conservador.")
print("  • A semântica dos 4-mers é INFERIDA por ordem de grandeza, não documentada")
print("    aqui. Os 4-mers sobrepostos não são independentes, então a previsão de")
print("    composição é otimista. A receita de geração pertence aos Methods.")

print("\n" + "=" * 100); print("FIM DA CÉLULA 1"); print("=" * 100)

# =============================================================================
# ===== FIM CÉLULA 1 ==========================================================
# =============================================================================

ONDE O SINAL SE PERDE ENTRE 5.000 FRAGMENTOS E UM DIAGNÓSTICO
Auditoria de Multiple Instance Learning em fragmentômica de cfDNA — revisão 2
Assinatura da configuração : 6cc8b31519897578
Repetições independentes   : 5 sementes
Coorte externa             : exploratória, tarefa distinta
Verificações extras        : desligadas
Hardware                   : cuda

----------------------------------------------------------------------------------------------------
AMBIENTE DE SOFTWARE
----------------------------------------------------------------------------------------------------
  python           3.12.13
  numpy            2.0.2
  pandas           2.3.3
  scikit-learn     1.6.1
  torch            2.10.0+cu128
  seaborn          0.13.2
  gpu              Tesla T4

----------------------------------------------------------------------------------------------------
ASSINATURA DOS ARQUIVOS DE ENTRADA (SHA-256)
----------------------------------------------------------------------------------


### Auditoria decomposta de *Multiple Instance Learning* e *Foundation Models* em fragmentômica de *cfDNA*

---

## 1. O problema

No plasma de qualquer pessoa circulam milhões de fragmentos curtos de DNA
livre, o ***cfDNA***, liberados por células que morreram em qualquer tecido do
corpo. Quando existe um tumor, uma fração desses fragmentos veio dele. Em
doença inicial, essa fração fica abaixo de 1%.

Fragmentos de origem tumoral têm propriedades físicas distintas. São em média
mais curtos, terminam em posições diferentes do genoma e se distribuem de
maneira desigual pelos cromossomos. A fragmentômica explora essas diferenças
para detectar câncer a partir de uma amostra de sangue.

Um modelo de linguagem genômica — um ***Foundation Model*** — pode ler a
sequência de cada fragmento e produzir um vetor de 256 números. Entre esses
vetores e um diagnóstico existe uma cadeia de decisões de engenharia: como
combinar milhares de fragmentos num único escore, onde aplicar as não
linearidades, que classificador usar, quantas moléculas amostrar por paciente.


---

## 2. As quatro perguntas

**Onde o sinal se perde?**
Construímos uma escada de seis degraus ligando o classificador mais simples ao
mais complexo, mudando **um elemento por vez**. Cada degrau recebe a sua fatia
exata da diferença total.

**O encoder semântico acrescenta alguma coisa?**
Tentamos reconstruir o vetor do *Foundation Model* a partir de duas coisas
elementares: a composição de bases dos fragmentos e o lugar onde caíram no
genoma. O que sobra é o candidato ao que o pré-treinamento acrescentou.

**Quanta profundidade cada representação exige?**
Uma segunda escada, de dois degraus, compara a mesma medida de cobertura
calculada sobre todas as moléculas disponíveis e sobre as mesmas cinco mil que
alimentam os braços neurais. Um só elemento muda: o número de moléculas.


---

## 3. O regime de sinal fraco

Este é o ponto que torna o resto do caderno legível.

Um fragmento isolado separa muito pouco doentes de saudáveis. A diferença é da
ordem de centésimos de desvio padrão. Só a média sobre milhares de moléculas
torna o sinal visível.

Nesse regime, uma não linearidade aplicada **antes** da média opera em boa
parte sobre variação não informativa. Uma função ReLU zera todos os valores
negativos. Sobre ruído aproximadamente simétrico, ela corta metade da
distribuição para um lado só e altera a média antes que o cancelamento
aconteça.

```
ruído simétrico          ReLU ANTES da média          ReLU DEPOIS da média
┌───────────┐            ┌─────┐                      ┌───────────┐
─┤   ▄█▄   ├─  →         ┤  █▄ ├  média deslocada     ┤   ▄█▄   ├  média
└───────────┘            └─────┘                      └───────────┘  preservada
     0                        0                            0
```

Isto não vale para qualquer função nem em qualquer regime. É uma afirmação
sobre **estes dados**, e o degrau chamado ORDEM a mede diretamente.

---

## 4. Desenho amostral

| Partição              | Coorte                             |   n | Papel                                                  |
| :-------------------- | :--------------------------------- | --: | :----------------------------------------------------- |
| **Desenvolvimento**   | Baltimore (Cristiano et al., 2019) | 537 | Validação cruzada de cinco dobras; toda a análise forense |
| **Avaliação externa** | Hong Kong (Jiang et al., 2015)     | 225 | Inferência apenas; estatuto exploratório               |

O treinamento interno é um detector **pan-câncer**: 276 casos de oito tipos
tumorais contra 261 controles saudáveis. **Baltimore não contém nenhum caso de
carcinoma hepatocelular.**

A avaliação externa testa **carcinoma hepatocelular** (90 casos) contra
saudáveis (32). Hepatite B (67) e cirrose (36) não entram no cálculo da AUROC.
Aparecem numa figura própria, para mostrar o que cada modelo faz diante de
inflamação hepática sem tumor.

Quando o desempenho cai na coorte externa, **três causas se confundem**: tipo
tumoral não visto no treinamento, ascendência (ocidental e asiática) e
protocolo de sequenciamento. Não é validação externa da mesma tarefa clínica, e
sim **generalização para uma coorte não homóloga**. O resultado primário deste
caderno é **interno**.

As análises forenses usam exclusivamente Baltimore. Hong Kong só é aberta
depois que todos os modelos estão treinados.

---

## 5. As representações comparadas

Cada paciente é descrito por uma **bolsa** de 5.000 fragmentos reais. O que
muda entre os braços é o que medimos sobre cada molécula.

| Representação                    |   Dim. | O que é, em palavras simples                                               | Estrutura |
| :------------------------------- | -----: | :------------------------------------------------------------------------- | :-------- |
| **Caduceus**                     |    256 | o que o *Foundation Model* «vê» na sequência ao redor do fragmento         | densa     |
| **4-mers**                       |    256 | com que frequência cada combinação de quatro letras aparece                | densa     |
| **Comprimento**                  |      1 | o tamanho da molécula, em pares de base                                    | densa     |
| **Perfil de *end-motif***        |    256 | as quatro bases no ponto exato onde a enzima cortou                        | contagem  |
| **Cobertura em 5 Mb, painel**    |  1.614 | quantos fragmentos curtos e longos há em cada janela de 5 milhões de bases | contagem  |
| **Cobertura em 5 Mb, promotora** |  1.698 | o mesmo, restrito às janelas promotoras                                    | contagem  |
| **Ocupação de promotores**       | 39.614 | em qual promotor cada fragmento caiu                                       | contagem  |

A distinção entre **representação densa** e **representação por contagem** é a
chave de um dos resultados. Uma representação por contagem divide as moléculas
em caixas, e precisa de moléculas suficientes em cada caixa. Uma representação
densa não tem caixas.

Um detalhe que costuma confundir: a cobertura restrita a promotores tem **mais**
colunas que a cobertura no painel inteiro, não menos. A razão é que o limiar
mínimo de extensão acessível exigido por janela é diferente nas duas contagens.
Restringir a região admite janelas que antes eram descartadas por cobertura
insuficiente.

A ocupação de promotores é 88% zeros: com 5.000 fragmentos e 39.614 alvos, a
maioria fica vazia. Ela entra na reconstrução do encoder e no teste de coorte,
não como classificador.

---

## 6. Procedência e preparação dos dados

Os dados de entrada provêm de sequenciamentos genômicos totais de baixa
cobertura, padronizados no repositório FinaleDB.

1. **Seleção e amostragem.** A partir dos alinhamentos contra a referência
`hg38`, selecionaram-se moléculas cujo ponto médio cai dentro de um painel
curado de 39.614 regiões promotoras humanas e a até 1 kb do centro de um
promotor. Usar o ponto médio, e não a sobreposição, evita que uma molécula
seja contada em duas regiões vizinhas. São elegíveis os fragmentos com
qualidade de mapeamento mínima de 30 e comprimento entre 100 e 220 pares de
base. Cada paciente é representado por exatamente 5.000 fragmentos reais,
sorteados por amostragem de reservatório, **sem preenchimento artificial**.

2. ***Embeddings* do Caduceus.** Para cada fragmento, extraiu-se a sequência da
referência ao longo de sua extensão acrescida de ±6 pares de base além dos
sítios de clivagem, formando uma janela mediana de 178 pares de base. As
sequências passaram pelo codificador bidirecional congelado *Caduceus-Ph*.
O vetor de cada fragmento é a **média da camada final sobre as posições da
janela**: a ordem interna é colapsada antes de qualquer agregação entre
fragmentos. Fragmentos mapeados na fita negativa são revertidos e
complementados antes da codificação.

3. **Composição de 4-mers.** Frequência relativa dos 256 tetranucleotídeos ao
longo da mesma janela.

4. **Cobertura em janelas de 5 Mb.** Razões entre fragmentos curtos e longos em
janelas autossômicas de cinco milhões de bases, normalizadas pela extensão
de painel acessível em cada janela. Calculada em três versões, descritas na
secção 7.

5. **Perfil de *end-motif*.** Calculado neste caderno, a partir das coordenadas
e da referência.

As quatro bases do ponto de corte ocupam 2,2% da janela lida pelo encoder.
O *end-motif* **não** é medido pelas representações de sequência; é medido
separadamente.

---

## 7. As duas escadas

Cada degrau muda exatamente uma coisa. A soma dos degraus iguala a diferença
entre os extremos por construção aritmética, o que **verifica a integridade do
cálculo**, e não estabelece causa.

### Escada principal — seis degraus

Liga o classificador mais simples ao *Attention-MIL* completo, sobre a
representação do *Foundation Model*.

```
Caduceus (Média → Logística)
        │ COMPRIMENTO   acrescenta o comprimento do fragmento
        │ NORMALIZAÇÃO  padronização por paciente → por fragmento
        │ CLASSIFICADOR troca a regressão logística por uma rede
        │ PROJEÇÃO      acrescenta uma projeção DEPOIS da média
        │ ORDEM         move a mesma projeção para ANTES da média
        │ AGREGADOR     troca a média por atenção aprendida
Caduceus + Comprimento (Projeção → Atenção MIL)
```

Uma camada afim comuta com a média, para quaisquer pesos:

`Linear(média(x)) = W·média(x) + b = média(W·x + b) = média(Linear(x))`

Um portão verifica essa identidade numericamente. Por isso as duas arquiteturas
do degrau ORDEM diferem **exclusivamente na posição da ReLU**.

### Escada da cobertura — dois degraus

Liga a cobertura calculada sobre todas as moléculas do painel à cobertura
calculada sobre a mesma bolsa das representações neurais.

```
Cobertura em 5 Mb, painel inteiro, todos os fragmentos
        │ REGIÃO        restringe as janelas às regiões promotoras
        │ PROFUNDIDADE  restringe aos mesmos 5.000 fragmentos da bolsa
Cobertura em 5 Mb, promotora, os mesmos 5.000
```

O degrau **PROFUNDIDADE é a comparação mais limpa de todo o caderno**: mesma
região, mesmas janelas, mesmo classificador, mesmas partições, mesmas sementes.
Só o número de moléculas contadas muda.

O degrau REGIÃO não é igualmente limpo. Restringir a região altera, por
consequência, quais janelas têm cobertura suficiente para entrar no vetor.
Isso é declarado, não corrigido: a alternativa seria comparar janelas que não
existem nas duas condições.

As duas escadas são medidas em duas famílias de classificador, rede e regressão
logística, para que nenhuma conclusão dependa de uma escolha de modelo.

---

## 8. O perfil de *end-motif*

O ***end-motif*** são as quatro bases no ponto exato onde a enzima cortou a
molécula de *cfDNA*. Enzimas diferentes cortam em contextos diferentes, e essa
preferência muda em doença. É um sinal estabelecido na literatura, e **totalmente
independente** de tudo o que as demais representações medem.

```
Fita de referência   5'--[ s  s+1  s+2  s+3  .................  e-1 ]--3'
Fita complementar    3'--[ ...................................  e-1 ]--5'

Extremo de MONTANTE : bases em s, s+1, s+2, s+3, lidas para a frente
Extremo de JUSANTE  : bases em e-1, e-2, e-3, e-4, complementadas
```

Uma molécula de fita dupla tem **dois** extremos 5′, um em cada ponta. Os dois
são contados, o de jusante revertido e complementado.

**Por que dá para calcular sem as leituras brutas.** Sabendo a coordenada do
corte e o genoma de referência, lemos as quatro bases ali. Elas diferem das
bases reais do paciente apenas quando ele carrega uma variante naquelas quatro
posições, cerca de 0,4% dos casos. A preferência enzimática aparece em **onde**
os cortes acontecem, não em quais letras o paciente tem. É por isso que o
perfil é recuperável de coordenadas.

**Duas coisas são medidas.** A primeira é a forma do perfil: se motivos
associados a uma enzima conhecida aparecerem enriquecidos acima do acaso, a
extração de coordenadas está correta. A segunda é a precisão: dividimos a bolsa
ao meio e perguntamos se as duas metades ordenam os pacientes da mesma forma.
Se não ordenam, o perfil é ruidoso nesta profundidade, e isso limita o que se
pode concluir dele.

O caderno também projeta, pela fórmula de Spearman-Brown, quantos fragmentos
seriam necessários para atingir precisões maiores, e confronta essa projeção
com quantos fragmentos elegíveis cada paciente realmente possui.

---

## 9. Cinco repetições, e duas formas de incerteza

Redes treinadas sobre bolsas de 5.000 instâncias com sinal fraco têm objetivos
de alta variância: a mesma arquitetura, com outra semente, pode chegar a outro
resultado. Todo o *pipeline* é executado **cinco vezes**, com partições e
inicializações independentes, produzindo 25 comparações por degrau.

Cada degrau é reportado por **dois estimadores diferentes**, cada um com a sua
incerteza:

| Estimador                          | Incerteza             | O que varia                | O que fica fixo      |
| :--------------------------------- | :-------------------- | :------------------------- | :------------------- |
| Média das diferenças **por dobra** | desvio entre sementes | partições e inicializações | os pacientes         |
| Diferença do **escore agrupado**   | bootstrap pareado     | a amostra de pacientes     | os modelos treinados |

Eles **não medem a mesma quantidade**. A escada soma o primeiro; o segundo é um
teste independente sobre os mesmos pacientes. Quando divergem, a divergência é
informação: significa que as réplicas discordam entre si.

Medimos também a **concordância entre réplicas**, pela correlação de Spearman
média par a par entre os escores que as cinco repetições atribuem aos mesmos
pacientes. Ela responde a uma pergunta que a AUROC média não responde: as
réplicas ordenam os pacientes da mesma forma, ou cada uma inventa a sua própria
ordenação?

---

## 10. Portões de qualidade

A execução é interrompida se qualquer um falhar. A ideia é falhar em segundos,
e não descobrir o problema depois de meia hora de treinamento.

| Portão                                | O que protege                                                                                              |
| :------------------------------------ | :--------------------------------------------------------------------------------------------------------- |
| **Consistência estrutural**           | Formas e tipos de todos os arquivos                                                                        |
| **Linhagem das coordenadas**          | As coordenadas descrevem os mesmos fragmentos que geraram os *embeddings*                                  |
| **Finitude**                          | Completa nas coberturas e comprimentos; triagem amostral nos *embeddings*                                  |
| **Coberturas distintas**              | Os três arquivos de cobertura não são cópias um do outro                                                   |
| **Geometria pareada**                 | As duas coberturas promotoras têm as mesmas janelas, senão o degrau PROFUNDIDADE mudaria mais de uma coisa |
| **Sem preenchimento artificial**      | Bolsas incompletas corromperiam médias e pesos de atenção                                                  |
| **Vetor idêntico**                    | O tensor consumido pela regressão e pela rede é o mesmo objeto                                             |
| **Comutação linear/média**            | A camada afim comuta com a média; o degrau ORDEM isola a posição da ReLU                                   |
| **Escala do controle de canal único** | O canal de comprimento está padronizado em todos os braços                                                 |
| **Identidade telescópica**            | Nas duas escadas, a soma dos degraus iguala a diferença entre extremos                                     |
| **Substrato do resíduo**              | Se a reconstrução linear falhar, a análise do resíduo é abortada                                           |
| **Perfis normalizados**               | Todo perfil de *end-motif* soma exatamente um                                                              |

Avisos de convergência **não são silenciados**. Um modelo que não convergiu é
informação, não ruído. São contados e relatados.

---

## 11. O que o caderno grava

Toda figura e toda tabela citável saem daqui. **Nenhum valor é transcrito à
mão** para o texto final.

**Figuras**

| Arquivo                    | Conteúdo                                                          |
| :------------------------- | :---------------------------------------------------------------- |
| `Figura_1_Escada`          | A escada principal e a sua replicação noutra representação        |
| `Figura_2_Generalizacao`   | Desempenho de cada braço nas duas coortes                         |
| `Figura_3_Grupos_Clinicos` | Escores por grupo clínico, incluindo doença hepática sem tumor    |
| `Figura_4_Mecanismo`       | Por que a ordem entre não linearidade e agregação importa         |
| `Figura_5_Profundidade`    | A escada da cobertura e o apetite por dados de cada representação |
| `Figura_6_EndMotif`        | O perfil de motivos e a sua precisão nesta profundidade           |

**Tabelas** — desempenho interno e externo, ponto de operação, separabilidade
de coorte, diagnóstico da atenção, controles de otimização, as duas escadas,
AUROC por semente e os perfis de *end-motif* por paciente.

**`procedencia.json`** — a assinatura SHA-256 da configuração, do código e de
cada arquivo de entrada; o resultado de cada portão de integridade; e todos os
números reportados. É este arquivo que torna o texto auditável.

---

## 12. Como executar

**Configuração no Kaggle**

- **Accelerator:** GPU T4 ou P100. Obrigatória.
- **Internet:** **ON**. Uma das células instala a biblioteca que lê o genoma de
referência no formato `.2bit`.
- **Persistence:** Files only.

**Datasets a anexar**

| Dataset                               | Conteúdo                                                                                      |
| :------------------------------------ | :-------------------------------------------------------------------------------------------- |
| `cfdna-promoter-multimodal-resume-v3` | As representações por paciente, as coordenadas dos fragmentos e as contagens de elegibilidade |
| `cfdna-ref-hg38`                      | `hg38.2bit`, a referência genômica                                                            |

**Tempo esperado:** cerca de quarenta minutos. A leitura do genoma para o
perfil de motivos consome entre sete e oito minutos; o restante é treinamento
e reamostragem.

**Ordem das células.** O perfil de *end-motif* é calculado **antes** de as
bolsas subirem para a memória da placa gráfica, para que os dois picos de
consumo de memória não coincidam. As células devem ser executadas na ordem em
que aparecem.

Não escrevemos um leitor próprio do formato `.2bit` de propósito. Um erro sutil
na decodificação produziria bases erradas sem emitir aviso, e contaminaria todo
o perfil de motivos silenciosamente.

---

## 13. O que este caderno não estabelece

Cada célula termina com a sua própria lista de limites. Os que valem para o
conjunto:

- **A decomposição depende do caminho.** Outra ordem dos degraus produziria
outras fatias com a mesma soma. A identidade telescópica verifica o cálculo,
não estabelece causalidade.

- **As 25 comparações não são independentes.** Os conjuntos de treinamento se
sobrepõem e os pacientes se repetem. A contagem de sinais é descritiva.

- **A coorte externa não é homóloga.** A queda confunde tipo tumoral,
ascendência e protocolo, que este conjunto de dados não separa. Não é
validação clínica de espécie alguma.

- **O resíduo e a sua distribuição nula são lineares.** Sinal residual não
identifica o pré-treinamento como causa. Ausência de sinal não excluiria
estrutura não linear.

- **Os máximos foram escolhidos depois de ver os dados.** Com um número grande
de braços, o vencedor carrega viés de seleção. As sementes descrevem a
instabilidade; não a removem.

- **O braço «Somente Comprimento» não é um controle nulo.** É um controle de
canal único, e o comprimento carrega sinal real.

- **Buscas amplas de hiperparâmetros não foram executadas**, e poderiam
favorecer arquiteturas com mais parâmetros.

- **A correspondência entre a linha *i* de cada arquivo e o paciente *i*** é
pressuposto da etapa de geração das representações. Aqui apenas a dimensão é
verificada.

In [2]:
# =============================================================================
# ===== CÉLULA 2 — REPRESENTAÇÕES E CONTEÚDO DO ENCODER =======================
# =============================================================================

print("=" * 100)
print("CÉLULA 2 — REPRESENTAÇÕES E CONTEÚDO DO ENCODER")
print("=" * 100)

MAPA = {"healthy": 0, "cirrhosis": 0, "hepatitis_b": 0,
        "bile_duct_cancer": 1, "breast_cancer": 1, "colorectal_cancer": 1,
        "duodenal_cancer": 1, "gastric_cancer": 1, "lung_cancer": 1,
        "ovarian_cancer": 1, "pancreatic_cancer": 1, "liver_cancer": 1}
sem_mapa = sorted(set(META["condicao"].dropna()) - set(MAPA))
if sem_mapa:
    raise RuntimeError(f"Condições clínicas sem rótulo definido: {sem_mapa}")
rotulos = META["condicao"].map(MAPA).astype(np.int8).to_numpy()

Y_DEV = rotulos[IDX_DEV].astype(int)
N_DEV = len(IDX_DEV)
hcc_dev = int((META.loc[IDX_DEV, "condicao"] == HCC).sum())

print(f"\n  Desenvolvimento : {N_DEV} pacientes "
      f"({Y_DEV.sum()} casos / {(1-Y_DEV).sum()} controles)")
print(f"  Externa         : {len(IDX_EXT)} pacientes")
print(f"  Carcinoma hepatocelular no treino : {hcc_dev}")
if hcc_dev == 0:
    print("  As duas tarefas diferem: treino pan-câncer, avaliação num tipo tumoral")
    print("  ausente do treino, noutra população e noutro protocolo.")

# -----------------------------------------------------------------------------
# Vetores por paciente.
#
# A leitura em blocos ordena os índices e depois desfaz a ordenação, para que
# cada linha continue correspondendo ao seu paciente. As linhas 0..N_DEV-1 são
# Baltimore; de N_DEV em diante, Hong Kong.
# -----------------------------------------------------------------------------
print("\n  Construindo vetores por paciente...")
t0 = time.time()
ordem    = np.concatenate([IDX_DEV, IDX_EXT])
ord_sort = np.sort(ordem)
inversa  = np.argsort(np.argsort(ordem))

POS = np.full(N_PAC, -1, dtype=np.int64)
POS[ordem] = np.arange(len(ordem))
if not (POS[ordem] >= 0).all() or not np.array_equal(ord_sort[inversa], ordem):
    raise RuntimeError("Falha de alinhamento entre índices de paciente e linhas.")
if not np.array_equal(ordem[:N_DEV], IDX_DEV) or not np.array_equal(ordem[N_DEV:], IDX_EXT):
    raise RuntimeError("As linhas a partir de N_DEV não correspondem à coorte externa.")

def media_por_paciente(matriz, lote=40):
    buf = np.zeros((len(ord_sort), matriz.shape[2]), dtype=np.float32)
    for i in range(0, len(ord_sort), lote):
        idx = ord_sort[i:i+lote]
        buf[i:i+len(idx)] = np.asarray(matriz[idx], dtype=np.float32).mean(axis=1)
    return buf[inversa]

V_CAD  = media_por_paciente(CAD)
V_KMER = media_por_paciente(KMER)
COMP_ORD = COMP[ord_sort].astype(np.float32)[inversa]
V_COMP   = COMP_ORD.mean(axis=1, keepdims=True)

BINS = np.arange(CFG["COMPRIMENTO_MIN_HIST"], CFG["COMPRIMENTO_MAX_HIST"] + 1, 3)
validos   = COMP_ORD[COMP_ORD > 0]
frac_fora = float(np.mean((validos < BINS[0]) | (validos > BINS[-1])))
V_HIST = np.stack([np.histogram(l[l > 0], bins=BINS, density=True)[0]
                   for l in COMP_ORD]).astype(np.float32)

prom_ord = np.asarray(PROM[ord_sort], dtype=np.int64)[inversa]
V_OCUP = np.zeros((len(ordem), N_PROM), dtype=np.float32)
for i, linha in enumerate(prom_ord):
    np.add.at(V_OCUP[i], linha[linha >= 0], 1.0)
V_OCUP /= np.maximum(V_OCUP.sum(axis=1, keepdims=True), 1.0)
del prom_ord

V_DELFI              = DELFI[ordem]
V_DELFI_PROM_ALL     = DELFI_PROM_ALL[ordem]
V_DELFI_PROM_MATCHED = DELFI_PROM_MATCHED[ordem]

frac_zeros = float((V_OCUP == 0).mean())
print(f"  concluído em {time.time()-t0:.0f} s")

if frac_fora > CFG["FRACAO_MAX_FORA_HIST"]:
    raise RuntimeError("Descarte excessivo de fragmentos fora do histograma.")
print(f"  Fragmentos fora do histograma de comprimento : {100*frac_fora:.4f}%")
print(f"  Zeros na matriz de ocupação de promotores    : {100*frac_zeros:.1f}%")

# -----------------------------------------------------------------------------
# QUANTAS MOLÉCULAS CAEM EM CADA CÉLULA DE CONTAGEM
#
# Uma representação por contagem divide as moléculas em caixas. Se houver poucas
# moléculas por caixa, o valor de cada caixa é ruído de Poisson, independente de
# qualquer biologia. Este é o número que explica, adiante, por que a cobertura
# restrita e o perfil de motivos se comportam como se comportam.
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("MOLÉCULAS POR CÉLULA DE CONTAGEM")
print("A densidade de contagem é o que limita uma representação discretizada.")
print("-" * 100)
n_bins_painel = DELFI.shape[1] // 3
n_bins_prom   = DELFI_PROM_ALL.shape[1] // 3
DENSIDADE = {
    "Cobertura promotora, os mesmos 5.000":
        CFG["N_FRAGMENTOS"] / n_bins_prom,
    "Ocupação de promotores":
        CFG["N_FRAGMENTOS"] / N_PROM,
}
print(f"  {'representação':<44s} {'células':>10s} {'moléculas/célula':>18s}")
print(f"  {'Cobertura promotora, os mesmos 5.000':<44s} {n_bins_prom:>10d} "
      f"{CFG['N_FRAGMENTOS']/n_bins_prom:>18.1f}")
print(f"  {'Ocupação de promotores':<44s} {N_PROM:>10d} "
      f"{CFG['N_FRAGMENTOS']/N_PROM:>18.3f}")
print(f"  (a densidade do perfil de end-motif entra na Célula 3; as densidades")
print(f"   da cobertura completa não são calculáveis aqui, porque o número de")
print(f"   fragmentos usados não foi gravado por paciente.)")

# -----------------------------------------------------------------------------
# Partições, definidas uma vez e compartilhadas por tudo: reconstrução do
# resíduo, permutações e treinamento.
#
# O conjunto de parada é estratificado. Ele é consultado a cada época para
# escolher o melhor ponto do treino; uma divisão aleatória poderia entregar
# 25 casos contra 39 controles em vez de 32/32, acrescentando ruído justamente
# à quantidade usada para seleção.
# -----------------------------------------------------------------------------
def constroi_dobras(semente, y=None):
    y = Y_DEV if y is None else y
    skf = StratifiedKFold(n_splits=CFG["N_DOBRAS"], shuffle=True, random_state=semente)
    dobras = []
    for itr, iva in skf.split(np.zeros(len(y)), y):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=CFG["FRACAO_PARADA"],
                                     random_state=semente + len(dobras))
        i_aj, i_par = next(sss.split(np.zeros(len(itr)), y[itr]))
        dobras.append(dict(ajuste=itr[i_aj], parada=itr[i_par], validacao=iva))
    return dobras

DOBRAS_POR_SEMENTE = {s: constroi_dobras(s) for s in CFG["SEMENTES"]}
_d0 = DOBRAS_POR_SEMENTE[SEMENTE_MESTRA][0]
print(f"\n  Por dobra: ajuste {len(_d0['ajuste'])}, parada {len(_d0['parada'])}, "
      f"validação {len(_d0['validacao'])}")
print(f"  Proporção de casos — ajuste {Y_DEV[_d0['ajuste']].mean():.3f}, "
      f"parada {Y_DEV[_d0['parada']].mean():.3f}, "
      f"validação {Y_DEV[_d0['validacao']].mean():.3f}")

# -----------------------------------------------------------------------------
# QUANTO SINAL EXISTE POR FRAGMENTO, E QUANTO POR PACIENTE
#
# O «d de Cohen» mede a distância entre dois grupos em unidades de desvio
# padrão: 0,2 é um efeito pequeno, 0,8 é grande. Calculamos duas vezes,
# tratando cada FRAGMENTO como observação, e cada PACIENTE (a média dos seus
# 5.000) como observação, usando exatamente os mesmos pacientes nos dois
# níveis. É a medida que justifica a existência do Multiple Instance Learning:
# nenhum fragmento sozinho decide.
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("QUANTO SINAL EXISTE POR FRAGMENTO E POR PACIENTE")
print("-" * 100)
rng_d = np.random.default_rng(SEMENTE_MESTRA)
casos_loc = np.where(Y_DEV == 1)[0]; ctrl_loc = np.where(Y_DEV == 0)[0]
n_d = min(len(casos_loc), len(ctrl_loc), CFG["N_PACIENTES_POR_CLASSE_D"])
sel_caso = np.sort(rng_d.choice(casos_loc, n_d, replace=False))
sel_ctrl = np.sort(rng_d.choice(ctrl_loc, n_d, replace=False))

def cohen_d(a, b):
    return np.abs(a.mean(0) - b.mean(0)) / (
        np.sqrt((a.var(0, ddof=1) + b.var(0, ddof=1)) / 2) + 1e-12)

d_pac_v = cohen_d(V_CAD[sel_caso], V_CAD[sel_ctrl])
bp = np.asarray(CAD[np.sort(IDX_DEV[sel_caso])], np.float32).reshape(-1, D_EMB)
bn = np.asarray(CAD[np.sort(IDX_DEV[sel_ctrl])], np.float32).reshape(-1, D_EMB)
d_frag_v = cohen_d(bp, bn)
del bp, bn

D_FRAG, D_PAC = float(np.median(d_frag_v)), float(np.median(d_pac_v))
print(f"  Amostra (a mesma nos dois níveis) : {n_d} casos + {n_d} controles")
print(f"  Por fragmento isolado : d = {D_FRAG:.4f}   (mediana das {D_EMB} dimensões)")
print(f"  Por paciente          : d = {D_PAC:.4f}   (mediana das {D_EMB} dimensões)")

# -----------------------------------------------------------------------------
# O ENCODER CONTÉM ALGO ALÉM DE COMPOSIÇÃO E LOCALIZAÇÃO?
#
# Tentamos reconstruir os 256 números do Caduceus a partir de duas coisas
# simples: as contagens de 4-mers (composição de bases) e a ocupação de
# promotores (onde o fragmento caiu). O R² diz quanto da variação reproduzimos:
# R² = 1 é reconstrução perfeita.
#
# O resíduo é o candidato ao que o pré-treino acrescentou. Se ele discriminar
# casos de controles, há sinal além da composição.
#
# O resíduo é reconstruído DENTRO DE CADA RÉPLICA, com as dobras daquela
# réplica: «cinco réplicas independentes» seria falso se elas partilhassem a
# construção da feature.
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("O ENCODER CONTÉM ALGO ALÉM DE COMPOSIÇÃO E LOCALIZAÇÃO?")
print("=" * 100)

def constroi_residuo(dobras, com_pca_padronizada=False):
    """Devolve o resíduo fora-da-dobra e os R² de cada reconstrução."""
    resid = np.zeros_like(V_CAD)
    r2 = dict(kmer=[], ocup=[], ambos=[], ocup_pad=[]); alphas = []
    for dobra in dobras:
        tr, va = dobra["ajuste"], dobra["validacao"]
        pca = PCA(n_components=CFG["N_COMPONENTES_OCUP"], svd_solver="randomized",
                  random_state=SEMENTE_MESTRA).fit(V_OCUP[tr])
        Z = pca.transform(V_OCUP)
        blocos = [("kmer", V_KMER), ("ocup", Z), ("ambos", np.hstack([V_KMER, Z]))]
        if com_pca_padronizada:
            pp = Pipeline([("esc", StandardScaler()),
                           ("pca", PCA(n_components=CFG["N_COMPONENTES_OCUP"],
                                       svd_solver="randomized",
                                       random_state=SEMENTE_MESTRA))]).fit(V_OCUP[tr])
            blocos.append(("ocup_pad", pp.transform(V_OCUP)))
        for etiq, Xe in blocos:
            Xe = np.ascontiguousarray(Xe, dtype=np.float32)
            mod = Pipeline([("esc", StandardScaler()),
                            ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 13)))]).fit(
                                Xe[tr], V_CAD[tr])
            prev = mod.predict(Xe[va])
            r2[etiq].append(r2_score(V_CAD[va], prev, multioutput="variance_weighted"))
            if etiq == "ambos":
                resid[va] = V_CAD[va] - prev
                # Para a coorte externa, o resíduo é a média das cinco dobras.
                resid[N_DEV:] += (V_CAD[N_DEV:] - mod.predict(Xe[N_DEV:])) / len(dobras)
                alphas.append(float(np.ravel(mod.named_steps["ridge"].alpha_)[0]))
    return resid, r2, alphas

t0 = time.time()
RESIDUOS, R2_TODOS = {}, dict(kmer=[], ocup=[], ambos=[], ocup_pad=[])
for i_s, s in enumerate(CFG["SEMENTES"]):
    resid, r2, alphas = constroi_residuo(
        DOBRAS_POR_SEMENTE[s],
        com_pca_padronizada=(i_s == 0 and CFG["VERIFICACOES_EXTRA"]))
    RESIDUOS[s] = resid
    for k in r2:
        R2_TODOS[k].extend(r2[k])
    if i_s == 0:
        ALPHAS_S1 = alphas

R2_KMER  = float(np.mean(R2_TODOS["kmer"]))
R2_OCUP  = float(np.mean(R2_TODOS["ocup"]))
R2_RESID = float(np.mean(R2_TODOS["ambos"]))
R2_OCUP_PAD = float(np.mean(R2_TODOS["ocup_pad"])) if R2_TODOS["ocup_pad"] else float("nan")

print(f"\n  Quanto do vetor do Caduceus é reconstruível (R² fora da dobra, média de")
print(f"  {len(R2_TODOS['ambos'])} dobras = {len(CFG['SEMENTES'])} sementes × {CFG['N_DOBRAS']} dobras):")
print(f"    só composição de 4-mers     : {R2_KMER:.3f}")
print(f"    só ocupação de promotores   : {R2_OCUP:.3f}")
if np.isfinite(R2_OCUP_PAD):
    print(f"      (com PCA padronizada)     : {R2_OCUP_PAD:.3f}")
print(f"    as duas juntas              : {R2_RESID:.3f}   "
      f"[{min(R2_TODOS['ambos']):.3f}–{max(R2_TODOS['ambos']):.3f}]")
print(f"  Regularização escolhida (semente 1): {', '.join(f'{a:.3g}' for a in ALPHAS_S1)}")
print(f"  ({time.time()-t0:.0f} s)")

RESID_OK       = R2_RESID >= CFG["R2_ABORTA_RESIDUO"]
RESID_INTERPRE = R2_RESID >= CFG["R2_INTERPRETA_RESIDUO"]
if not RESID_OK:
    print(f"\n  ANÁLISE DO RESÍDUO ABORTADA: a reconstrução falhou (R² = {R2_RESID:.3f}).")
    print(f"  Nesse caso o resíduo é praticamente o vetor original, e a sua AUROC")
    print(f"  não diz nada sobre o pré-treino.")

# -----------------------------------------------------------------------------
# DISTRIBUIÇÃO NULA EMPÍRICA
#
# Embaralhamos os rótulos e refazemos o procedimento. A AUROC que um modelo
# atinge sobre rótulos aleatórios não é exatamente 0,500: com 537 pacientes e
# 256 dimensões, ela flutua. Esta é a distribuição dessa flutuação.
#
# Três exigências: a nula usa a MESMA representação, o MESMO classificador e a
# MESMA estrutura de replicação (5 sementes × 5 dobras) da estatística observada.
#
# Sobre as partições. Quando a representação NÃO depende da partição, as dobras
# são refeitas nos rótulos permutados (`reestratifica=True`), para que a
# partição não fique condicionada ao desfecho verdadeiro. Quando a representação
# FOI CONSTRUÍDA fora-da-dobra sob certas partições — o caso do resíduo —,
# refazê-las descasaria a feature das dobras e o pareamento fora-da-dobra se
# perderia justamente no nulo, inflando-o. Aí mantemos as originais.
# -----------------------------------------------------------------------------
def distribuicao_nula(X_por_semente, n_perm, reestratifica):
    nulos = []
    rngp = np.random.RandomState(SEMENTE_MESTRA)
    for _ in range(n_perm):
        y = rngp.permutation(Y_DEV)
        por_semente = []
        for s in CFG["SEMENTES"]:
            X = X_por_semente[s]; aucs = []
            dobras = constroi_dobras(s, y=y) if reestratifica else DOBRAS_POR_SEMENTE[s]
            for dobra in dobras:
                tr, va = dobra["ajuste"], dobra["validacao"]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", ConvergenceWarning)
                    m = Pipeline([("e", StandardScaler()),
                                  ("l", LogisticRegression(max_iter=1000))]).fit(X[tr], y[tr])
                aucs.append(roc_auc_score(y[va], m.predict_proba(X[va])[:, 1]))
            por_semente.append(np.mean(aucs))
        nulos.append(np.mean(por_semente))
    return np.array(nulos)

print("\n" + "-" * 100)
print(f"DISTRIBUIÇÃO NULA EMPÍRICA DO RESÍDUO ({CFG['N_PERMUTACOES']} embaralhamentos × "
      f"{len(CFG['SEMENTES'])} sementes × {CFG['N_DOBRAS']} dobras)")
print("-" * 100)
t0 = time.time()
if RESID_OK:
    NULOS_RESID = distribuicao_nula(RESIDUOS, CFG["N_PERMUTACOES"], reestratifica=False)
    PERC95_RESID = float(np.percentile(NULOS_RESID, 95))
    print(f"  Mediana {np.median(NULOS_RESID):.3f} · percentil 95 {PERC95_RESID:.3f}   "
          f"({time.time()-t0:.0f} s)")
else:
    NULOS_RESID, PERC95_RESID = None, float("nan")

if CFG["VERIFICACOES_EXTRA"]:
    t0 = time.time()
    NULOS_CAD = distribuicao_nula({s: V_CAD for s in CFG["SEMENTES"]},
                                  CFG["N_PERMUTACOES"], reestratifica=True)
    PERC95_CAD = float(np.percentile(NULOS_CAD, 95))
    print(f"  [extra] Vetor completo do Caduceus — mediana {np.median(NULOS_CAD):.3f} · "
          f"percentil 95 {PERC95_CAD:.3f}   ({time.time()-t0:.0f} s)")
else:
    PERC95_CAD = float("nan")

# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 100)
print("  • O resíduo depende da classe de reconstrução escolhida aqui, linear.")
print("    Ausência de sinal sob um modelo linear não prova ausência de informação.")
print("  • A distribuição nula também é linear: não cobre estrutura não linear,")
print("    nem o patamar mais alto de redes com atenção e seleção de época.")
print(f"  • Com {CFG['N_PERMUTACOES']} permutações, o menor p atingível é "
      f"{1/(CFG['N_PERMUTACOES']+1):.4f}.")
print("  • O nulo do resíduo usa as partições que o geraram. Refazê-las descasaria")
print("    a feature das dobras e inflaria o nulo — a alternativa seria pior.")
print("  • O d de Cohen mediano por dimensão é uma descrição do regime, não uma")
print("    avaliação preditiva por fragmento: não exclui dimensões informativas.")

print("\n" + "=" * 100); print("FIM DA CÉLULA 2"); print("=" * 100)

# =============================================================================
# ===== FIM CÉLULA 2 ==========================================================
# =============================================================================

CÉLULA 2 — REPRESENTAÇÕES E CONTEÚDO DO ENCODER

  Desenvolvimento : 537 pacientes (276 casos / 261 controles)
  Externa         : 225 pacientes
  Carcinoma hepatocelular no treino : 0
  As duas tarefas diferem: treino pan-câncer, avaliação num tipo tumoral
  ausente do treino, noutra população e noutro protocolo.

  Construindo vetores por paciente...
  concluído em 15 s
  Fragmentos fora do histograma de comprimento : 0.0000%
  Zeros na matriz de ocupação de promotores    : 88.4%

----------------------------------------------------------------------------------------------------
MOLÉCULAS POR CÉLULA DE CONTAGEM
A densidade de contagem é o que limita uma representação discretizada.
----------------------------------------------------------------------------------------------------
  representação                                   células   moléculas/célula
  Cobertura promotora, os mesmos 5.000                566                8.8
  Ocupação de promotores                            

In [3]:
# =============================================================================
# ===== CÉLULA 3 — PERFIL DE END-MOTIF E TETO DE PROFUNDIDADE =================
# =============================================================================
#
# POR QUE ESTA CÉLULA EXISTE
#
# Todos os resultados deste caderno são negativos ou nulos. Um resultado
# negativo pode vir de um defeito de extração tanto quanto de uma propriedade
# dos dados, e o leitor não tem como distinguir os dois casos.
#
# O perfil de end-motif resolve isso. É um sinal biológico conhecido,
# independente de tudo o que medimos até aqui, e recuperável das mesmas
# coordenadas. Se ele aparecer com a forma que a literatura descreve, a
# extração está correta.
#
# O QUE É O END-MOTIF
#
# São as quatro bases no ponto exato onde a enzima cortou a molécula. Enzimas
# diferentes cortam em contextos diferentes, e essa preferência muda em doença.
#
# POR QUE DÁ PARA CALCULAR SEM AS LEITURAS BRUTAS
#
# Sabendo a coordenada do corte e o genoma de referência, lemos as quatro bases
# ali. Elas diferem das bases reais do paciente apenas quando ele carrega uma
# variante naquelas quatro posições, cerca de 0,4% dos casos. A preferência
# enzimática aparece em ONDE os cortes acontecem, não em quais letras o
# paciente tem.
#
# ATENÇÃO: esta célula exige Internet ON, porque instala py2bit.
# =============================================================================

print("=" * 100)
print("CÉLULA 3 — PERFIL DE END-MOTIF E TETO DE PROFUNDIDADE")
print("=" * 100)

# -----------------------------------------------------------------------------
# py2bit lê o genoma no formato .2bit, que é como a referência está guardada.
# Não faz parte da imagem do Kaggle.
#
# Não escrevemos um leitor próprio de .2bit de propósito: um erro sutil na
# decodificação produziria bases erradas sem emitir aviso, e contaminaria todo
# o perfil de motivos silenciosamente.
# -----------------------------------------------------------------------------
try:
    import py2bit
    print("\n  py2bit já disponível")
except ModuleNotFoundError:
    print("\n  Instalando py2bit…", flush=True)
    _r = subprocess.run(f"{sys.executable} -m pip install -q py2bit",
                        shell=True, capture_output=True, text=True)
    if _r.returncode != 0:
        raise SystemExit(
            "Falha ao instalar py2bit.\n"
            "Verifique Settings > Internet: ON.\n"
            f"Saída do pip:\n{_r.stderr[-1500:]}")
    import py2bit
    print("  py2bit instalado")

ARQ_HG38 = acha("hg38.2bit")
print(f"  Referência : {ARQ_HG38}")

AUTOSSOMOS = [f"chr{i}" for i in range(1, 23)]
INDICE_CROMO = {c: i for i, c in enumerate(AUTOSSOMOS)}

# -----------------------------------------------------------------------------
# Traduzir bases em números
#
# Cada base vira um dígito: A=0, C=1, G=2, T=3. Bases indefinidas viram -1 e são
# descartadas. Assim um tetranucleotídeo vira um número de 0 a 255, como se
# fosse um número de quatro dígitos na base 4 — o que permite contar os 256
# motivos de uma vez, em vez de comparar textos.
# -----------------------------------------------------------------------------
CODIGO_BASE = np.full(256, -1, np.int8)
for _i, _b in enumerate("ACGT"):
    CODIGO_BASE[ord(_b)] = _i
    CODIGO_BASE[ord(_b.lower())] = _i

_B = "ACGT"
NOME_MOTIVO = [_B[i // 64] + _B[(i // 16) % 4] + _B[(i // 4) % 4] + _B[i % 4]
               for i in range(256)]

G2 = py2bit.open(str(ARQ_HG38))
TAMANHOS = {c: int(n) for c, n in G2.chroms().items() if c in AUTOSSOMOS}
if len(TAMANHOS) != 22:
    G2.close()
    raise RuntimeError(f"Referência com {len(TAMANHOS)} autossomos, esperado 22")
print(f"  Genoma     : {len(TAMANHOS)} autossomos, "
      f"{sum(TAMANHOS.values())/1e9:.2f} bilhões de bases.")

COORDS = np.load(PASTA / ARQUIVOS["coordenadas"], mmap_mode="r")
print(f"  Coordenadas: forma {COORDS.shape}  tipo {COORDS.dtype}")

# -----------------------------------------------------------------------------
# GATE — linhagem das coordenadas
#
# Confirma que estas coordenadas descrevem os mesmos fragmentos que geraram os
# embeddings. Sem isso, os dois conjuntos podem descrever moléculas diferentes
# sem que nada acuse.
# -----------------------------------------------------------------------------
if COORDS.ndim != 3 or COORDS.shape[0] != N_PAC \
        or COORDS.shape[1] != CFG["N_FRAGMENTOS"] or COORDS.shape[2] != 4:
    G2.close()
    raise RuntimeError(
        f"Coordenadas com forma {COORDS.shape}; esperado "
        f"({N_PAC}, {CFG['N_FRAGMENTOS']}, 4).")

ARQ_LAYOUT = acha("extraction_layout.json", obrigatorio=False)
if ARQ_LAYOUT:
    _layout = json.loads(Path(ARQ_LAYOUT).read_text())
    if int(_layout.get("n_samples", -1)) != N_PAC:
        G2.close()
        raise RuntimeError(f"layout diz {_layout.get('n_samples')} amostras, "
                           f"metadata diz {N_PAC}")
    if int(_layout.get("promoter_pool", -1)) != CFG["N_FRAGMENTOS"]:
        G2.close()
        raise RuntimeError(f"pool divergente: {_layout.get('promoter_pool')}")
    PROTOCOLO_ORIGEM = str(_layout.get("protocol_hash", "não informado"))
    print(f"  Linhagem confere — protocolo de origem {PROTOCOLO_ORIGEM[:16]}…")
else:
    PROTOCOLO_ORIGEM = "extraction_layout.json ausente"
    print(f"  AVISO: extraction_layout.json não encontrado. A correspondência")
    print(f"  entre estas coordenadas e os embeddings não pôde ser verificada.")

# -----------------------------------------------------------------------------
# Como os dois extremos são lidos
#
#   Fita de referência   5'--[ s s+1 s+2 s+3 ................ e-1 ]--3'
#   Fita complementar    3'--[ ............................... e-1 ]--5'
#
#   Extremo de MONTANTE : bases em s, s+1, s+2, s+3, lidas para a frente.
#   Extremo de JUSANTE  : bases em e-1, e-2, e-3, e-4, complementadas —
#                         porque o 5' daquela fita aponta na direção oposta.
#
# A coluna de fita das coordenadas NÃO é usada aqui. Ela indica qual leitura
# veio primeiro na biblioteca, o que é outra coisa: os dois extremos são
# extremos 5' legítimos da molécula de fita dupla, e a convenção do campo é
# contar os dois.
# -----------------------------------------------------------------------------
def conta_motivos(codigos, inicios, fins, ambos):
    n_max = len(codigos)
    contagem = np.zeros(256, np.int64)
    usados = 0

    dentro = (inicios >= 0) & (inicios + 4 <= n_max)
    s = inicios[dentro]
    if len(s):
        b0, b1 = codigos[s], codigos[s + 1]
        b2, b3 = codigos[s + 2], codigos[s + 3]
        ok = (b0 >= 0) & (b1 >= 0) & (b2 >= 0) & (b3 >= 0)
        if ok.any():
            idx = (((b0[ok].astype(np.int32) * 4 + b1[ok]) * 4 + b2[ok]) * 4 + b3[ok])
            contagem += np.bincount(idx, minlength=256)
            usados += int(ok.sum())

    # jusante, revertido e complementado (o complemento de um código é 3 menos ele)
    if ambos:
        dentro = (fins - 4 >= 0) & (fins <= n_max)
        e = fins[dentro]
        if len(e):
            d0, d1 = codigos[e - 4], codigos[e - 3]
            d2, d3 = codigos[e - 2], codigos[e - 1]
            ok = (d0 >= 0) & (d1 >= 0) & (d2 >= 0) & (d3 >= 0)
            if ok.any():
                r0 = 3 - d3[ok].astype(np.int32)
                r1 = 3 - d2[ok].astype(np.int32)
                r2 = 3 - d1[ok].astype(np.int32)
                r3 = 3 - d0[ok].astype(np.int32)
                idx = (((r0 * 4 + r1) * 4 + r2) * 4 + r3)
                contagem += np.bincount(idx, minlength=256)
                usados += int(ok.sum())
    return contagem, usados

# -----------------------------------------------------------------------------
# Percorremos por cromossomo, não por paciente.
#
# Carregar o cromossomo uma vez e atender todos os pacientes nele evita manter
# o genoma inteiro na memória. O pico fica em torno de 750 MB, no chr1. Esta
# célula roda ANTES de as bolsas subirem para a GPU, para que os dois picos de
# memória não coincidam.
# -----------------------------------------------------------------------------
TODAS_COORD = np.asarray(COORDS)      # ~60 MB, cabe inteiro
CONTAGENS = np.zeros((N_PAC, 256), np.int64)
USADOS    = np.zeros(N_PAC, np.int64)

METADE = CFG["N_FRAGMENTOS"] // 2
if CFG["MOTIF_TESTE_METADES"]:
    CONT_A = np.zeros((N_PAC, 256), np.int64)
    CONT_B = np.zeros((N_PAC, 256), np.int64)

print(f"\n  Varrendo {len(AUTOSSOMOS)} cromossomos para {N_PAC} pacientes…")
t0 = time.time()
for _ic, c in enumerate(AUTOSSOMOS, start=1):
    seq = G2.sequence(c, 0, TAMANHOS[c])
    codigos = CODIGO_BASE[np.frombuffer(seq.encode("ascii", "replace"), np.uint8)]
    del seq
    ci = INDICE_CROMO[c]
    for i in range(N_PAC):
        m = TODAS_COORD[i, :, 0] == ci
        if not m.any():
            continue
        cont, usa = conta_motivos(codigos, TODAS_COORD[i, m, 1].astype(np.int64),
                                  TODAS_COORD[i, m, 2].astype(np.int64),
                                  CFG["MOTIF_AMBOS_EXTREMOS"])
        CONTAGENS[i] += cont
        USADOS[i] += usa
        if CFG["MOTIF_TESTE_METADES"]:
            pos = np.where(m)[0]
            for alvo, sel in ((CONT_A, pos[pos < METADE]), (CONT_B, pos[pos >= METADE])):
                if len(sel):
                    c2, _ = conta_motivos(
                        codigos, TODAS_COORD[i, sel, 1].astype(np.int64),
                        TODAS_COORD[i, sel, 2].astype(np.int64),
                        CFG["MOTIF_AMBOS_EXTREMOS"])
                    alvo[i] += c2
    del codigos
    print(f"    {c:<6s} concluído   ({(time.time()-t0)/60:.1f} min acumulados)", flush=True)

G2.close()
del TODAS_COORD
print(f"  Varredura concluída em {(time.time()-t0)/60:.1f} min")

# -----------------------------------------------------------------------------
# GATES do perfil
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("GATE — PERFIS DE END-MOTIF")
print("-" * 100)
sem_cortes = int((USADOS == 0).sum())
if sem_cortes:
    raise RuntimeError(f"{sem_cortes} pacientes sem nenhum corte utilizável")

cortes_esperados = CFG["N_FRAGMENTOS"] * (2 if CFG["MOTIF_AMBOS_EXTREMOS"] else 1)
perda_borda = 1.0 - USADOS / cortes_esperados
print(f"  Cortes por paciente — mínimo {int(USADOS.min()):,} · "
      f"mediana {int(np.median(USADOS)):,} · esperado {cortes_esperados:,}")
print(f"  Perda por borda de cromossomo ou base indefinida : "
      f"{100*perda_borda.max():.4f}% no pior caso")
if perda_borda.max() > 0.01:
    print(f"  AVISO: mais de 1% dos extremos foi descartado nalgum paciente.")

MOTIF = (CONTAGENS / np.maximum(USADOS[:, None], 1)).astype(np.float32)
if not np.isfinite(MOTIF).all():
    raise RuntimeError("Valores não finitos no perfil de end-motif")
_soma = MOTIF.sum(axis=1)
if not np.allclose(_soma, 1.0, atol=1e-5):
    raise RuntimeError(f"Perfis não somam 1: {_soma.min():.6f} a {_soma.max():.6f}")
print(f"  Aprovado: todos os {N_PAC} perfis somam 1,000 e são finitos.")

# Reordenado para a convenção do resto do caderno: Baltimore primeiro.
V_MOTIF = MOTIF[ordem]

# -----------------------------------------------------------------------------
# CONFIABILIDADE POR DIVISÃO AO MEIO
#
# Dividimos a bolsa ao meio e perguntamos se as duas metades ordenam os
# pacientes da mesma forma. Se ordenam, o perfil é estável e a bolsa é
# suficiente. Se não, o perfil é ruidoso, e isso limita o que se pode concluir.
#
# Este teste não introduz nenhum fragmento novo, e por isso não reintroduz a
# diferença de profundidade amostral que o degrau PROFUNDIDADE mede adiante.
# -----------------------------------------------------------------------------
if CFG["MOTIF_TESTE_METADES"]:
    pa = CONT_A / np.maximum(CONT_A.sum(1, keepdims=True), 1)
    pb = CONT_B / np.maximum(CONT_B.sum(1, keepdims=True), 1)
    # Só motivos presentes: os raros são zero nas duas metades e não informam.
    presentes = np.where(MOTIF.mean(axis=0) > 1e-4)[0]
    _rs = [spearmanr(pa[:, j], pb[:, j]).correlation for j in presentes]
    _rs = np.array([r for r in _rs if np.isfinite(r)])
    CONF_METADE = float(np.median(_rs))
    # Spearman-Brown: a confiabilidade da bolsa inteira é maior que a de metade
    # dela, porque tem o dobro das observações.
    CONF_BOLSA = (2 * CONF_METADE / (1 + CONF_METADE)
                  if CONF_METADE > -1 else float("nan"))
    print("\n" + "-" * 100)
    print(f"CONFIABILIDADE POR DIVISÃO AO MEIO ({len(presentes)} motivos presentes)")
    print("-" * 100)
    print(f"  Concordância entre metades de {METADE:,} : {CONF_METADE:.3f}")
    print(f"  Estimada para a bolsa de {CFG['N_FRAGMENTOS']:,}     : {CONF_BOLSA:.3f}")
    if CONF_BOLSA >= 0.90:
        print(f"  Os {CFG['N_FRAGMENTOS']:,} fragmentos estimam o perfil de forma estável.")
    elif CONF_BOLSA >= 0.75:
        print(f"  Estabilidade moderada. A limitação precisa ser declarada.")
    else:
        print(f"  Perfil instável nesta profundidade. Diferenças pequenas entre")
        print(f"  grupos não são interpretáveis. O perfil serve como controle")
        print(f"  positivo da extração, não como representação candidata.")
else:
    CONF_METADE = CONF_BOLSA = float("nan")

# -----------------------------------------------------------------------------
# O QUE O PERFIL DIZ
#
# A literatura associa motivos iniciados em CC à clivagem por DNASE1L3. Se o
# enriquecimento aparecer, o perfil captou preferência enzimática, e não apenas
# a composição de bases das regiões amostradas.
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("PERFIL MÉDIO DA COORTE — CONTROLE POSITIVO DA EXTRAÇÃO")
print("-" * 100)
motif_medio = MOTIF.mean(axis=0)
ordem_motif = np.argsort(motif_medio)[::-1]
print(f"  {'motivo':>8s} {'frequência':>12s}")
for j in ordem_motif[:10]:
    print(f"  {NOME_MOTIVO[j]:>8s} {100*motif_medio[j]:11.3f}%")

FRAC_CC = float(sum(motif_medio[j] for j in range(256)
                    if NOME_MOTIVO[j].startswith("CC")))
print(f"\n  Fração de motivos iniciados em CC : {100*FRAC_CC:.1f}%")
print(f"  Sob cortes aleatórios seriam       : {100*16/256:.1f}%")
print(f"  Enriquecimento                     : {FRAC_CC/(16/256):.1f}x")
TOP_MOTIVOS = [NOME_MOTIVO[j] for j in ordem_motif[:10]]

# Diversidade: entropia normalizada. Perfis mais uniformes têm valor mais alto.
with np.errstate(divide="ignore", invalid="ignore"):
    _ent = -np.sum(np.where(MOTIF > 0, MOTIF * np.log(MOTIF), 0.0), axis=1)
DIVERSIDADE = (_ent / np.log(256)).astype(np.float32)
print(f"\n  Diversidade de motivos (entropia normalizada):")
print(f"    mediana {np.median(DIVERSIDADE):.4f} · "
      f"faixa {DIVERSIDADE.min():.4f}–{DIVERSIDADE.max():.4f}")

MOTIF_POR_CELULA = cortes_esperados / 256.0
print(f"\n  Cortes por motivo, em média : {MOTIF_POR_CELULA:.0f}")
print(f"  É esta densidade que limita a confiabilidade medida acima.")

# -----------------------------------------------------------------------------
# TETO DE PROFUNDIDADE
#
# A confiabilidade medida vale para a bolsa de 5.000. A pergunta seguinte é
# quantos fragmentos elegíveis cada paciente TEM disponíveis, se abandonássemos
# a bolsa fixa.
#
# Spearman-Brown invertido: se a bolsa atual de B fragmentos rende
# confiabilidade r, o fator k que atinge uma meta r* é
#
#        k = r*(1 - r) / (r(1 - r*))
#
# A fórmula pressupõe que os fragmentos adicionais são equivalentes aos atuais.
# Fragmentos de um mesmo paciente compartilham região e protocolo, então a
# projeção é otimista.
# -----------------------------------------------------------------------------
ARQ_CONTAGENS = acha("promoter_fragment_counts.csv", obrigatorio=False)
PROJECOES = []
TETO = {}
if ARQ_CONTAGENS is not None and np.isfinite(CONF_BOLSA):
    CONT_FRAG = pd.read_csv(ARQ_CONTAGENS, dtype={"sample_id": str})
    faltando = {"sample_id", "estudo", "n_promoter", "n_all"} - set(CONT_FRAG.columns)
    if faltando:
        raise RuntimeError(f"Colunas ausentes em promoter_fragment_counts.csv: {faltando}")
    ids_meta = set(META["sample_id"].astype(str).str.strip())
    CONT_FRAG["sample_id"] = CONT_FRAG["sample_id"].astype(str).str.strip()
    ausentes = ids_meta - set(CONT_FRAG["sample_id"])
    if ausentes:
        raise RuntimeError(
            f"{len(ausentes)} pacientes do estudo não aparecem nas contagens. "
            f"Exemplos: {sorted(ausentes)[:5]}")
    CONT_FRAG = CONT_FRAG[CONT_FRAG["sample_id"].isin(ids_meta)].copy()

    print("\n" + "-" * 100)
    print("FRAGMENTOS ELEGÍVEIS DISPONÍVEIS POR PACIENTE")
    print("A coluna n_promoter é o universo de uma reextração restrita a promotores;")
    print("n_all é o universo maior, que exigiria mudar a regra de seleção.")
    print("-" * 100)
    for coluna, etiqueta in (("n_promoter", "promotor"), ("n_all", "painel inteiro")):
        print(f"\n  {etiqueta}")
        print(f"  {'coorte':<26s} {'n':>5s} {'mínimo':>10s} {'quartil 1':>11s} "
              f"{'mediana':>10s} {'quartil 3':>11s} {'máximo':>10s}")
        for coorte, g in list(CONT_FRAG.groupby("estudo")) + [("TODOS", CONT_FRAG)]:
            v = g[coluna].to_numpy()
            print(f"  {str(coorte):<26s} {len(v):>5d} {int(v.min()):>10,d} "
                  f"{int(np.percentile(v, 25)):>11,d} {int(np.median(v)):>10,d} "
                  f"{int(np.percentile(v, 75)):>11,d} {int(v.max()):>10,d}")

    n_prom_disp = CONT_FRAG["n_promoter"].to_numpy()

    def fragmentos_para(meta, r, base):
        if not (0.0 < r < 1.0) or meta >= 1.0:
            return float("inf")
        k = meta * (1.0 - r) / (r * (1.0 - meta))
        return k * base

    def confiabilidade_de(n, r, base):
        k = n / base
        return k * r / (1 + (k - 1) * r)

    print("\n" + "-" * 100)
    print("PROFUNDIDADE NECESSÁRIA E RETENÇÃO DE PACIENTES")
    print(f"Partindo da confiabilidade medida acima: {CONF_BOLSA:.3f} com "
          f"{CFG['N_FRAGMENTOS']:,} fragmentos.")
    print("-" * 100)
    print(f"  {'meta':>6s} {'fragmentos':>13s} {'fator':>8s} "
          f"{'pacientes que alcançam':>24s} {'retenção':>10s}")
    for meta in CFG["MOTIF_METAS"]:
        preciso = fragmentos_para(meta, CONF_BOLSA, CFG["N_FRAGMENTOS"])
        alcancam = int((n_prom_disp >= preciso).sum())
        retencao = alcancam / len(n_prom_disp)
        PROJECOES.append(dict(meta=meta, fragmentos=preciso,
                              alcancam=alcancam, retencao=retencao))
        print(f"  {meta:>6.2f} {preciso:>13,.0f} {preciso/CFG['N_FRAGMENTOS']:>7.1f}x "
              f"{alcancam:>18d}/{len(n_prom_disp):<5d} {100*retencao:>9.1f}%")

    p_ret = float(np.percentile(n_prom_disp, 100 * (1 - CFG["RETENCAO_MINIMA"])))
    r_ret = confiabilidade_de(p_ret, CONF_BOLSA, CFG["N_FRAGMENTOS"])
    TETO = dict(bolsa_maxima_com_retencao=p_ret, confiabilidade_nessa_bolsa=r_ret,
                mediana_n_promoter=float(np.median(n_prom_disp)),
                mediana_n_all=float(np.median(CONT_FRAG["n_all"].to_numpy())))

    print("\n" + "-" * 100)
    print("DECISÃO SOBRE REEXTRAIR O PERFIL COM BOLSA MAIOR")
    print("-" * 100)
    print(f"  Bolsa máxima que retém {100*CFG['RETENCAO_MINIMA']:.0f}% da coorte : "
          f"{p_ret:,.0f} fragmentos")
    print(f"  Confiabilidade projetada nessa bolsa      : {r_ret:.3f}")
    viaveis = [d for d in PROJECOES if d["retencao"] >= CFG["RETENCAO_MINIMA"]]
    if viaveis:
        melhor = max(viaveis, key=lambda d: d["meta"])
        print(f"\n  Reextrair atinge confiabilidade {melhor['meta']:.2f} retendo "
              f"{100*melhor['retencao']:.1f}% da coorte.")
        print(f"  Recomendação: VIÁVEL, com bolsa de {melhor['fragmentos']:,.0f} fragmentos.")
    else:
        print(f"\n  Nenhuma meta de {min(CFG['MOTIF_METAS']):.2f} ou mais é alcançável")
        print(f"  retendo {100*CFG['RETENCAO_MINIMA']:.0f}% da coorte.")
        print(f"  Recomendação: NÃO REEXTRAIR. O perfil permanece controle positivo")
        print(f"  da extração, não representação candidata a classificação.")
else:
    print("\n  Arquivo promoter_fragment_counts.csv não encontrado, ou confiabilidade")
    print("  indisponível. A análise de teto de profundidade foi pulada.")

# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("LIMITES DO QUE ESTE PERFIL MEDE")
print("-" * 100)
print("  • As bases vêm da referência hg38, não do paciente. Diferem do motivo")
print("    real apenas quando há variante nas quatro posições do corte — cerca")
print("    de 0,4% dos casos.")
print("  • Os fragmentos foram selecionados por ter ponto médio a até 1 kb do")
print("    centro de um promotor. O perfil descreve clivagem em região promotora,")
print("    e não é comparável a perfis genômicos publicados.")
print("  • Extremidades irregulares e adições não modeladas não são visíveis em")
print("    coordenadas. Para isso seriam necessárias as leituras brutas.")
print("  • A projeção de Spearman-Brown supõe fragmentos adicionais equivalentes")
print("    aos atuais. Fragmentos do mesmo paciente são correlacionados, então a")
print("    profundidade necessária é subestimada.")

print("\n" + "=" * 100); print("FIM DA CÉLULA 3"); print("=" * 100)

# =============================================================================
# ===== FIM CÉLULA 3 ==========================================================
# =============================================================================

CÉLULA 3 — PERFIL DE END-MOTIF E TETO DE PROFUNDIDADE

  Instalando py2bit…
  py2bit instalado
  Referência : /kaggle/input/datasets/fredericoflores1807/cfdna-ref-hg38/hg38.2bit
  Genoma     : 22 autossomos, 2.88 bilhões de bases.
  Coordenadas: forma (762, 5000, 4)  tipo int32
  Linhagem confere — protocolo de origem d7761ac06840461a…

  Varrendo 22 cromossomos para 762 pacientes…
    chr1   concluído   (0.4 min acumulados)
    chr2   concluído   (0.7 min acumulados)
    chr3   concluído   (1.0 min acumulados)
    chr4   concluído   (1.3 min acumulados)
    chr5   concluído   (1.6 min acumulados)
    chr6   concluído   (1.8 min acumulados)
    chr7   concluído   (2.0 min acumulados)
    chr8   concluído   (2.2 min acumulados)
    chr9   concluído   (2.4 min acumulados)
    chr10  concluído   (2.6 min acumulados)
    chr11  concluído   (2.8 min acumulados)
    chr12  concluído   (3.0 min acumulados)
    chr13  concluído   (3.2 min acumulados)
    chr14  concluído   (3.3 min acumulados)

In [4]:
# =============================================================================
# ===== CÉLULA 4 — A ESCADA DE SEIS DEGRAUS ===================================
# =============================================================================

print("=" * 100)
print("CÉLULA 4 — A ESCADA DE SEIS DEGRAUS, REPETIDA EM CINCO SEMENTES")
print("=" * 100)

AGREGACAO = {"cad_att": "atencao", "cad_mean": "media_apos_projecao",
             "cad_vec_mlp": "media_antes_projecao", "cad_vec_mlpdir": "media_sem_projecao",
             "kmer_att": "atencao", "kmer_mean": "media_apos_projecao",
             "kmer_vec_mlp": "media_antes_projecao", "length": "atencao"}
FONTE = {"cad_att": "caduceus", "cad_mean": "caduceus", "cad_vec_mlp": "caduceus",
         "cad_vec_mlpdir": "caduceus", "kmer_att": "kmers", "kmer_mean": "kmers",
         "kmer_vec_mlp": "kmers", "length": "zeros",
         "delfi": "tabular", "delfi_prom_all": "tabular",
         "delfi_prom_matched": "tabular"}

# Braços tabulares: o modelo nunca vê fragmentos, só um vetor por paciente.
# Indexados por índice GLOBAL de paciente, como as matrizes originais.
TABULAR = {
    "delfi":              DELFI,
    "delfi_prom_all":     DELFI_PROM_ALL,
    "delfi_prom_matched": DELFI_PROM_MATCHED,
}

# Nestes modos o modelo nunca vê fragmentos individuais, só a média. Como a
# padronização é afim, média((x-μ)/σ) = (média(x)-μ)/σ, e a entrada pode ser
# derivada dos vetores já calculados. Um gate verifica essa igualdade.
SEM_BOLSA = {"media_antes_projecao", "media_sem_projecao"}

NEURAIS = ["cad_att", "cad_mean", "cad_vec_mlp", "cad_vec_mlpdir",
           "kmer_att", "kmer_mean", "kmer_vec_mlp", "length",
           "delfi", "delfi_prom_all", "delfi_prom_matched"]

LINEARES = [("cad_lin", "V_CAD"), ("cad_vec_lin", "X_CAD_COMP"),
            ("rep_kmer", "V_KMER"), ("rep_len", "V_HIST"),
            ("rep_delfi", "V_DELFI"),
            ("rep_delfi_prom_all", "V_DELFI_PROM_ALL"),
            ("rep_delfi_prom_matched", "V_DELFI_PROM_MATCHED"),
            ("rep_motif", "V_MOTIF")]

X_CAD_COMP = np.hstack([V_CAD, V_COMP]).astype(np.float32)
MATRIZ = {"V_CAD": V_CAD, "X_CAD_COMP": X_CAD_COMP, "V_KMER": V_KMER,
          "V_HIST": V_HIST, "V_DELFI": V_DELFI,
          "V_DELFI_PROM_ALL": V_DELFI_PROM_ALL,
          "V_DELFI_PROM_MATCHED": V_DELFI_PROM_MATCHED,
          "V_MOTIF": V_MOTIF}

print(f"\n  Braços neurais  : {len(NEURAIS)}")
print(f"  Braços lineares : {len(LINEARES)} (mais o resíduo, se computável)")


def ram_disponivel():
    try:
        with open("/proc/meminfo") as f:
            for linha in f:
                if linha.startswith("MemAvailable:"):
                    return int(linha.split()[1]) * 1024
    except Exception:
        pass
    return None

def sobe_em_blocos(mm, lote):
    dtype_torch = torch.from_numpy(np.empty(0, dtype=mm.dtype)).dtype
    t = torch.empty(tuple(mm.shape), dtype=dtype_torch, device=DISPOSITIVO)
    for i in range(0, mm.shape[0], lote):
        t[i:i+lote] = torch.from_numpy(np.ascontiguousarray(mm[i:i+lote])).to(DISPOSITIVO)
    return t

BOLSA_GPU = {}
LOTE_SUBIDA = 50
if CFG["BOLSAS_NA_GPU"] and torch.cuda.is_available():
    livre, _ = torch.cuda.mem_get_info()
    preciso = int((CAD.nbytes + KMER.nbytes + COMP.nbytes * 2) * 1.05)
    ram = ram_disponivel()
    pico_host = LOTE_SUBIDA * CAD.shape[1] * CAD.shape[2] * CAD.dtype.itemsize
    print(f"\n  Embeddings: {CAD.dtype}, {CAD.nbytes/2**30:.2f} GB cada · "
          f"GPU precisa {preciso/2**30:.2f} GB de {livre/2**30:.2f} GB livres")
    print(f"  Pico de RAM do host na subida: {pico_host/2**30:.3f} GB" +
          (f" de {ram/2**30:.2f} GB livres" if ram else ""))
    if preciso < livre * 0.60:
        BOLSA_GPU["caduceus"] = sobe_em_blocos(CAD, LOTE_SUBIDA)
        BOLSA_GPU["kmers"]    = sobe_em_blocos(KMER, LOTE_SUBIDA)
        # float32 aqui, igual ao caminho por disco: elimina divergência numérica
        # entre os dois caminhos. O tensor é pequeno.
        BOLSA_GPU["comp"]     = torch.from_numpy(COMP.astype(np.float32)).to(DISPOSITIVO)
        print(f"  Bolsas residentes na GPU.")
    else:
        print(f"  Memória insuficiente; leitura por disco (mais lenta).")
NA_GPU = bool(BOLSA_GPU)

def estatisticas(fonte, idx_ajuste):
    """Média e desvio de cada canal, calculados só no conjunto de treino."""
    s = sq = 0.0; nc = 0
    for i in range(0, len(idx_ajuste), 60):
        c = COMP[np.sort(idx_ajuste[i:i+60])].astype(np.float64)
        s += c.sum(); sq += (c**2).sum(); nc += c.size
    mu_c = s / nc; sd_c = np.sqrt(max(sq/nc - mu_c**2, 1e-12))
    if fonte == "zeros":
        e = dict(mu=np.zeros(D_EMB, np.float32), sigma=np.ones(D_EMB, np.float32),
                 mu_comp=np.float32(mu_c), sigma_comp=np.float32(sd_c))
    elif NA_GPU:
        idx = torch.as_tensor(np.sort(idx_ajuste), device=DISPOSITIVO, dtype=torch.long)
        soma = torch.zeros(D_EMB, dtype=torch.float64, device=DISPOSITIVO)
        somq = torch.zeros(D_EMB, dtype=torch.float64, device=DISPOSITIVO); n = 0
        for i in range(0, len(idx), 60):
            x = BOLSA_GPU[fonte][idx[i:i+60]].double()
            soma += x.sum(dim=(0, 1)); somq += (x**2).sum(dim=(0, 1))
            n += x.shape[0] * x.shape[1]
        mu = (soma/n); sd = (somq/n - mu**2).clamp_min(1e-12).sqrt()
        e = dict(mu=mu.float().cpu().numpy(), sigma=sd.float().cpu().numpy(),
                 mu_comp=np.float32(mu_c), sigma_comp=np.float32(sd_c))
    else:
        M = CAD if fonte == "caduceus" else KMER
        soma = np.zeros(D_EMB); somq = np.zeros(D_EMB); n = 0
        for i in range(0, len(idx_ajuste), 30):
            b = np.asarray(M[np.sort(idx_ajuste[i:i+30])], dtype=np.float64)
            soma += b.sum(axis=(0,1)); somq += (b**2).sum(axis=(0,1))
            n += b.shape[0]*b.shape[1]
        mu = soma/n; sd = np.sqrt(np.maximum(somq/n - mu**2, 1e-12))
        e = dict(mu=mu.astype(np.float32), sigma=sd.astype(np.float32),
                 mu_comp=np.float32(mu_c), sigma_comp=np.float32(sd_c))
    if NA_GPU:
        e["mu_t"]    = torch.as_tensor(e["mu"], device=DISPOSITIVO)
        e["sigma_t"] = torch.as_tensor(e["sigma"], device=DISPOSITIVO)
        e["muc_t"]   = torch.as_tensor(float(e["mu_comp"]), device=DISPOSITIVO)
        e["sdc_t"]   = torch.as_tensor(float(e["sigma_comp"]), device=DISPOSITIVO)
    return e

def monta_bolsa(fonte, idx_glob, est):
    """Bolsa padronizada (lote, 5000, 257) como tensor na GPU."""
    if NA_GPU:
        idx  = torch.as_tensor(idx_glob, device=DISPOSITIVO, dtype=torch.long)
        comp = (BOLSA_GPU["comp"][idx].unsqueeze(-1) - est["muc_t"]) / est["sdc_t"]
        if fonte == "zeros":
            base = torch.zeros(len(idx_glob), CFG["N_FRAGMENTOS"], D_EMB, device=DISPOSITIVO)
        else:
            base = (BOLSA_GPU[fonte][idx].float() - est["mu_t"]) / est["sigma_t"]
        return torch.cat([base, comp], dim=2)
    inv = np.argsort(np.argsort(idx_glob)); srt = np.sort(idx_glob)
    if fonte == "zeros":
        base = np.zeros((len(idx_glob), CFG["N_FRAGMENTOS"], D_EMB), np.float32)
    else:
        M = CAD if fonte == "caduceus" else KMER
        base = ((np.asarray(M[srt], dtype=np.float32)[inv] - est["mu"]) / est["sigma"])
    comp = ((COMP[srt].astype(np.float32)[inv][..., None] - est["mu_comp"])
            / est["sigma_comp"])
    return torch.from_numpy(np.ascontiguousarray(
        np.concatenate([base, comp], axis=2))).to(DISPOSITIVO)

def vetor_medio(fonte, est):
    """O mesmo vetor que a média da bolsa produziria, derivado dos agregados."""
    base = {"caduceus": V_CAD, "kmers": V_KMER}.get(
        fonte, np.zeros((len(ordem), D_EMB), np.float32))
    if fonte != "zeros":
        base = (base - est["mu"]) / est["sigma"]
    comp = (V_COMP - est["mu_comp"]) / est["sigma_comp"]
    return np.ascontiguousarray(np.hstack([base, comp]).astype(np.float32))

class ModeloBolsa(nn.Module):
    """
    Uma arquitetura, quatro modos de combinar os fragmentos:

      atencao               projeta cada fragmento e aprende um peso para cada um
      media_apos_projecao   projeta cada fragmento, depois tira a média
      media_antes_projecao  tira a média primeiro, depois projeta
      media_sem_projecao    tira a média e vai direto para a cabeça

    A projeção e a cabeça nascem antes do ramo de atenção, para que braços com
    o mesmo prefixo partam de pesos idênticos nessas camadas. Isso não vale para
    'media_sem_projecao', cuja cabeça tem outra forma e consome outra quantidade
    de números aleatórios — é outra arquitetura, por construção.
    """
    def __init__(self, d_in, d_oculta, modo):
        super().__init__()
        self.modo = modo
        if modo == "media_sem_projecao":
            self.projecao, d_cab = None, d_in
        else:
            self.projecao = nn.Sequential(nn.Linear(d_in, d_oculta), nn.ReLU())
            d_cab = d_oculta
        self.cabeca = nn.Sequential(
            nn.Linear(d_cab, d_oculta), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(d_oculta, d_oculta // 2), nn.ReLU(),
            nn.Linear(d_oculta // 2, 1))
        self.atencao = nn.Sequential(
            nn.Linear(d_oculta, d_oculta // 2), nn.Tanh(),
            nn.Linear(d_oculta // 2, 1)) if modo == "atencao" else None
        self.ultimos_pesos = None

    def forward(self, x):
        if self.modo == "atencao":
            h = self.projecao(x)
            w = torch.softmax(self.atencao(h), dim=1)
            self.ultimos_pesos = w.detach()
            z = (w * h).sum(dim=1)
        elif self.modo == "media_apos_projecao":
            z = self.projecao(x).mean(dim=1)
        else:
            v = x if x.dim() == 2 else x.mean(dim=1)
            z = v if self.projecao is None else self.projecao(v)
        return self.cabeca(z).squeeze(-1)

def curva_suavizada(trajetoria, janela):
    """
    A AUROC no conjunto de parada flutua mais entre épocas vizinhas do que o
    modelo realmente melhora. Escolher o máximo bruto é escolher ruído.
    Suavizamos com média móvel centrada antes de escolher a época e antes de
    aplicar o limiar da análise de sensibilidade.
    """
    a = np.asarray(trajetoria, float); n = len(a); m = janela // 2
    return np.array([a[max(0, i-m):min(n, i+m+1)].mean() for i in range(n)])

def treina(braco, dobra, est_dobra, semente):
    aj, par, val = dobra["ajuste"], dobra["parada"], dobra["validacao"]
    fonte = FONTE[braco]; modo = AGREGACAO.get(braco)
    est = est_dobra.get(fonte) if CFG["PADRONIZA_BOLSA"] else None
    sem_bolsa = modo in SEM_BOLSA
    V_AG = vetor_medio(fonte, est) if sem_bolsa else None
    tabular = braco in TABULAR

    torch.manual_seed(semente)
    if tabular:
        M_TAB = TABULAR[braco]
        modelo = nn.Sequential(
            nn.Linear(M_TAB.shape[1], CFG["DIM_OCULTA"]), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(CFG["DIM_OCULTA"], CFG["DIM_OCULTA"]//2), nn.ReLU(),
            nn.Linear(CFG["DIM_OCULTA"]//2, 1)).to(DISPOSITIVO)
        esc = StandardScaler().fit(M_TAB[IDX_DEV[aj]])
    else:
        M_TAB = None
        modelo = ModeloBolsa(D_EMB + 1, CFG["DIM_OCULTA"], modo).to(DISPOSITIVO)
        esc = None
    otim = torch.optim.AdamW(modelo.parameters(), lr=CFG["TAXA_APRENDIZADO"],
                             weight_decay=CFG["DECAIMENTO_PESO"])
    perda_fn = nn.BCEWithLogitsLoss()

    def passa(idx):
        if tabular:
            x = np.ascontiguousarray(esc.transform(M_TAB[idx]), dtype=np.float32)
            return modelo(torch.from_numpy(x).to(DISPOSITIVO)).squeeze(-1)
        ent = (torch.from_numpy(np.ascontiguousarray(V_AG[POS[idx]])).to(DISPOSITIVO)
               if sem_bolsa else monta_bolsa(fonte, idx, est))
        return modelo(ent)

    y_aj, y_par = Y_DEV[aj], Y_DEV[par]
    rngl = np.random.RandomState(semente)
    trajetoria, checkpoints = [], []
    melhor, sem_melhora = -1.0, 0
    n_cortes = n_passos = 0

    for epoca in range(1, CFG["MAX_EPOCAS"] + 1):
        modelo.train()
        perm = rngl.permutation(len(aj))
        for i in range(0, len(perm), CFG["TAMANHO_LOTE"]):
            sub = perm[i:i+CFG["TAMANHO_LOTE"]]
            logits = passa(IDX_DEV[aj[sub]])
            alvo = torch.from_numpy(np.ascontiguousarray(
                y_aj[sub], dtype=np.float32)).to(DISPOSITIVO)
            otim.zero_grad(set_to_none=True)
            perda_fn(logits, alvo).backward()
            nrm = float(nn.utils.clip_grad_norm_(modelo.parameters(), CFG["CLIP_GRADIENTE"]))
            n_passos += 1; n_cortes += int(nrm > CFG["CLIP_GRADIENTE"])
            otim.step()
        modelo.eval()
        with torch.no_grad():
            p = [torch.sigmoid(passa(IDX_DEV[par[i:i+CFG["TAMANHO_LOTE"]]])).cpu().numpy()
                 for i in range(0, len(par), CFG["TAMANHO_LOTE"])]
            a_par = roc_auc_score(y_par, np.concatenate(p))
        trajetoria.append(a_par)
        # Guardamos todas as épocas porque a escolha final usa a curva suavizada,
        # que só existe depois que o treino termina.
        checkpoints.append({k: v.detach().cpu().clone()
                            for k, v in modelo.state_dict().items()})
        if a_par > melhor + 1e-5:
            melhor, sem_melhora = a_par, 0
        else:
            sem_melhora += 1
            if epoca >= CFG["MIN_EPOCAS"] and sem_melhora >= CFG["PACIENCIA"]:
                break

    suave = curva_suavizada(trajetoria, CFG["JANELA_SUAVIZACAO"])
    i_suave, i_cru = int(np.argmax(suave)), int(np.argmax(trajetoria))
    modelo.load_state_dict(checkpoints[i_suave]); modelo.eval()
    del checkpoints

    def infere(idx_glob):
        out, pesos = [], []
        with torch.no_grad():
            for i in range(0, len(idx_glob), CFG["TAMANHO_LOTE"]):
                lote = idx_glob[i:i+CFG["TAMANHO_LOTE"]]
                out.append(torch.sigmoid(passa(lote)).cpu().numpy())
                if modo == "atencao" and getattr(modelo, "ultimos_pesos", None) is not None:
                    pesos.append(modelo.ultimos_pesos.squeeze(-1).cpu().numpy())
        return np.concatenate(out), (np.concatenate(pesos) if pesos else None)

    p_val, _ = infere(IDX_DEV[val])
    p_ext, w_ext = infere(IDX_EXT)
    del modelo; torch.cuda.empty_cache()
    return p_val, p_ext, w_ext, dict(
        epoca=i_suave + 1, epoca_crua=i_cru + 1, n_epocas=len(trajetoria),
        pico_parada_suave=float(np.max(suave)),
        pico_parada_cru=float(np.max(trajetoria)),
        cortes=n_cortes, passos=n_passos)

def ajusta_logistica(X, y, padroniza=True):
    passos = ([("esc", StandardScaler())] if padroniza else []) + \
             [("log", LogisticRegression(max_iter=3000, C=1.0))]
    with warnings.catch_warnings(record=True) as cap:
        warnings.simplefilter("always", ConvergenceWarning)
        m = Pipeline(passos).fit(X, y)
    return m, sum(1 for w in cap if issubclass(w.category, ConvergenceWarning))

# -----------------------------------------------------------------------------
# Uma réplica = um sorteio completo: novas partições, novas inicializações, e o
# resíduo reconstruído com as dobras desta réplica.
# -----------------------------------------------------------------------------
def executa_replica(semente, verbose=True):
    dobras = DOBRAS_POR_SEMENTE[semente]
    residuo_desta_replica = RESIDUOS[semente] if RESID_OK else None

    res = {k: dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[])
           for k, _ in LINEARES}
    res["cad_vec_lin_frag"] = dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[])
    for b in NEURAIS:
        res[b] = dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[],
                      epocas=[], epocas_cruas=[], pico_suave=[], pico_cru=[],
                      cortes=0, passos=0, pesos=[])
    if RESID_OK:
        res["rep_cad_resid"] = dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[])
    avisos = 0

    for i_d, dobra in enumerate(dobras, start=1):
        tr, va = dobra["ajuste"], dobra["validacao"]
        est_dobra = {f: estatisticas(f, IDX_DEV[tr]) for f in ("caduceus", "kmers", "zeros")}

        # --- braços lineares: padronização entre pacientes ---
        entradas = [(k, MATRIZ[m]) for k, m in LINEARES]
        if RESID_OK:
            entradas.append(("rep_cad_resid", residuo_desta_replica))
        for chave, X in entradas:
            m, nw = ajusta_logistica(X[tr], Y_DEV[tr]); avisos += nw
            pv = m.predict_proba(X[va])[:, 1]
            res[chave]["oof"][va] = pv
            res[chave]["auc_dobras"].append(roc_auc_score(Y_DEV[va], pv))
            res[chave]["ext"].append(m.predict_proba(X[N_DEV:])[:, 1])

        # --- o mesmo vetor, na escala de fragmento que a rede recebe ---
        Xf = vetor_medio("caduceus", est_dobra["caduceus"])
        m, nw = ajusta_logistica(Xf[tr], Y_DEV[tr], padroniza=False); avisos += nw
        pv = m.predict_proba(Xf[va])[:, 1]
        res["cad_vec_lin_frag"]["oof"][va] = pv
        res["cad_vec_lin_frag"]["auc_dobras"].append(roc_auc_score(Y_DEV[va], pv))
        res["cad_vec_lin_frag"]["ext"].append(m.predict_proba(Xf[N_DEV:])[:, 1])

        # --- braços neurais ---
        for b in NEURAIS:
            t0 = time.time()
            pv, pe, we, diag = treina(b, dobra, est_dobra, semente + 1000 * i_d)
            res[b]["oof"][va] = pv
            res[b]["auc_dobras"].append(roc_auc_score(Y_DEV[va], pv))
            res[b]["ext"].append(pe)
            res[b]["epocas"].append(diag["epoca"])
            res[b]["epocas_cruas"].append(diag["epoca_crua"])
            res[b]["pico_suave"].append(diag["pico_parada_suave"])
            res[b]["pico_cru"].append(diag["pico_parada_cru"])
            res[b]["cortes"] += diag["cortes"]; res[b]["passos"] += diag["passos"]
            if we is not None:
                res[b]["pesos"].append(we)
            if verbose:
                marca = ("" if diag["pico_parada_suave"] >= CFG["LIMIAR_PARADA"]
                         else "  [abaixo do limiar de parada]")
                print(f"    {nome(b):<66s} {res[b]['auc_dobras'][-1]:.3f}  "
                      f"ép {diag['epoca']:>2d}  ({time.time()-t0:.0f}s){marca}")

    for k in res:
        res[k]["auc_dobras"] = np.array(res[k]["auc_dobras"])
        res[k]["media"] = float(res[k]["auc_dobras"].mean())
        res[k]["externo"] = np.mean(res[k]["ext"], axis=0)
    return res, avisos

REPLICAS, AVISOS_CONV = [], 0
for i_s, semente in enumerate(CFG["SEMENTES"], start=1):
    print(f"\n{'='*100}\nRÉPLICA {i_s} de {len(CFG['SEMENTES'])} — semente {semente}\n{'='*100}")
    t0 = time.time()
    r, nw = executa_replica(semente); AVISOS_CONV += nw
    REPLICAS.append(r)
    print(f"  Réplica concluída em {(time.time()-t0)/60:.1f} min")

CHAVES = list(REPLICAS[0].keys())
# Escore ensemble por paciente: média das cinco réplicas. É o mesmo objeto usado
# pelas escadas, pelo ponto de operação e pela avaliação interna agregada.
OOF_ENS = {k: np.mean([r[k]["oof"] for r in REPLICAS], axis=0) for k in CHAVES}

# -----------------------------------------------------------------------------
# GATES numéricos
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("GATE — O VETOR PRÉ-CALCULADO É IGUAL À MÉDIA DA BOLSA?")
print("-" * 100)
est_v = {f: estatisticas(f, IDX_DEV[:300]) for f in ("caduceus", "kmers", "zeros")}
amostra = IDX_DEV[:8]; ERROS = {}
for f in ("caduceus", "kmers"):
    a1 = monta_bolsa(f, amostra, est_v[f]).mean(dim=1).cpu().numpy()
    a2 = vetor_medio(f, est_v[f])[POS[amostra]]
    ERROS[f] = float(np.abs(a1 - a2).max())
    print(f"  {f:<10s} diferença máxima : {ERROS[f]:.2e}")
    if ERROS[f] > 1e-4:
        raise RuntimeError(f"Os dois caminhos divergem para '{f}'; o degrau da "
                           f"cabeça não isolaria a cabeça.")
dp_nulo = float(monta_bolsa("zeros", amostra, est_v["zeros"])[:, :, -1].std().cpu())
print(f"  Controle de canal único, desvio do comprimento : {dp_nulo:.3f} (esperado ~1,0)")
if not (0.5 < dp_nulo < 2.0):
    raise RuntimeError("O canal de comprimento do controle de canal único não está padronizado.")
print("  Aprovado.")

# -----------------------------------------------------------------------------
# GATE — A CAMADA LINEAR COMUTA COM A MÉDIA?
#
#     Linear(média(x)) = W·média(x) + b = média(W·x + b) = média(Linear(x))
#
# A identidade vale para QUAISQUER pesos. Logo as duas ARQUITETURAS diferem
# exclusivamente na posição da ReLU.
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("GATE — A CAMADA LINEAR COMUTA COM A MÉDIA?")
print("-" * 100)
torch.manual_seed(SEMENTE_MESTRA)
_lin = nn.Linear(D_EMB + 1, CFG["DIM_OCULTA"]).to(DISPOSITIVO)
with torch.no_grad():
    _bolsa = monta_bolsa("caduceus", amostra, est_v["caduceus"])
    ERRO_COMUTA = float((_lin(_bolsa.mean(dim=1)) - _lin(_bolsa).mean(dim=1)).abs().max())
del _bolsa, _lin
print(f"  |Linear(média(x)) − média(Linear(x))| máximo : {ERRO_COMUTA:.2e}")
if ERRO_COMUTA > 1e-3:
    raise RuntimeError("A camada afim não comuta com a média; as duas arquiteturas "
                       "do degrau ORDEM difeririam em mais de uma coisa.")
print(f"  Aprovado. As arquiteturas 'Média→Projeção' e 'Projeção→Média' diferem")
print(f"  exclusivamente na posição da ReLU.")

# -----------------------------------------------------------------------------
# Funções compartilhadas pelas duas escadas
# -----------------------------------------------------------------------------
def atingiu_limiar(res_braco, i_fold):
    """A AUROC suavizada no conjunto de parada superou o limiar em alguma época.
    É uma triagem de desempenho mínimo, não um teste de convergência."""
    if "pico_suave" not in res_braco:
        return True                      # braços lineares não têm curva de parada
    return res_braco["pico_suave"][i_fold] >= CFG["LIMIAR_PARADA"]

def bootstrap_delta_auc(y, p_a, p_b, n_boot, semente):
    """Bootstrap pareado de pacientes, com os dois modelos fixos."""
    rng = np.random.default_rng(semente); n = len(y); d = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if np.unique(y[i]).size < 2:
            continue
        d.append(roc_auc_score(y[i], p_b[i]) - roc_auc_score(y[i], p_a[i]))
    d = np.asarray(d)
    return (float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))

def monta_escada(especificacao):
    """Calcula os dois estimadores de cada degrau de uma escada."""
    saida = []
    for orig, dest, etiq, desc in especificacao:
        por_semente = np.array([r[dest]["media"] - r[orig]["media"] for r in REPLICAS])
        por_dobra, ok = [], []
        for r in REPLICAS:
            for i in range(CFG["N_DOBRAS"]):
                por_dobra.append(r[dest]["auc_dobras"][i] - r[orig]["auc_dobras"][i])
                ok.append(atingiu_limiar(r[orig], i) and atingiu_limiar(r[dest], i))
        por_dobra, ok = np.array(por_dobra), np.array(ok)
        bmed, blo, bhi = bootstrap_delta_auc(Y_DEV, OOF_ENS[orig], OOF_ENS[dest],
                                             CFG["N_BOOTSTRAP"], SEMENTE_MESTRA)
        saida.append(dict(
            origem=orig, destino=dest, etiqueta=etiq, descricao=desc,
            delta=float(por_semente.mean()), dp_sementes=float(por_semente.std(ddof=1)),
            boot_delta=bmed, boot_inf=blo, boot_sup=bhi,
            neg=int((por_dobra < 0).sum()), n_comp=len(por_dobra), n_ok=int(ok.sum()),
            delta_ok=float(por_dobra[ok].mean()) if ok.any() else float("nan"),
            neg_ok=int((por_dobra[ok] < 0).sum()) if ok.any() else 0,
            delta_falha=float(por_dobra[~ok].mean()) if (~ok).any() else float("nan"),
            neg_falha=int((por_dobra[~ok] < 0).sum()) if (~ok).any() else 0))
    return saida

def verifica_telescopio(degraus_, primeiro, ultimo, rotulo):
    soma_ = sum(d["delta"] for d in degraus_)
    extremos_ = float(np.mean([r[ultimo]["media"] - r[primeiro]["media"] for r in REPLICAS]))
    discrep_ = abs(soma_ - extremos_)
    print(f"  Soma dos degraus            : {soma_:+.4f}")
    print(f"  Diferença entre os extremos : {extremos_:+.4f}")
    print(f"  Discrepância                : {discrep_:.2e}")
    if discrep_ > 1e-9:
        raise RuntimeError(f"A escada {rotulo} não fecha: erro de indexação ou de agregação.")
    print(f"  Aprovado: a decomposição é aritmeticamente exata.")
    return soma_, extremos_, discrep_

def imprime_escada(degraus_, titulo):
    print(f"\n  {'Degrau':<14s} {'O que muda':<48s} {'Δ':>8s} {'±dp':>7s} {'Sinal':>12s}")
    print("  " + "-" * 94)
    pior_ = min(degraus_, key=lambda d: d["delta"])
    for d in degraus_:
        marca = "  ← MAIOR PERDA" if d is pior_ else ""
        print(f"  {d['etiqueta']:<14s} {d['descricao']:<48s} "
              f"{d['delta']:+8.3f} {d['dp_sementes']:7.3f} "
              f"{d['neg']:>4d}/{d['n_comp']} neg{marca}")
    print("\n  " + "-" * 94)
    print(f"  {'Degrau':<14s} {'média das diferenças por dobra':>32s}   "
          f"{'diferença do escore agrupado':>38s}")
    print(f"  {'':14s} {'± dp entre sementes':>32s}   {'[IC 95% bootstrap pareado]':>38s}")
    print(f"  {'':14s} {'(pacientes fixos)':>32s}   {'(modelos fixos)':>38s}")
    print("  " + "-" * 94)
    for d in degraus_:
        e1 = f"{d['delta']:+.3f} ± {d['dp_sementes']:.3f}"
        e2 = f"{d['boot_delta']:+.3f}  [{d['boot_inf']:+.3f}, {d['boot_sup']:+.3f}]"
        cruza = "" if (d["boot_inf"] * d["boot_sup"]) > 0 else "  IC cruza zero"
        print(f"  {d['etiqueta']:<14s} {e1:>32s}   {e2:>38s}{cruza}")
    return pior_

# -----------------------------------------------------------------------------
# A ESCADA PRINCIPAL
#
# Cada degrau muda exatamente uma coisa. A soma dos seis iguala a diferença
# entre os extremos por construção aritmética — o que verifica a integridade do
# cálculo, não estabelece causa.
# -----------------------------------------------------------------------------
ESCADA = [
    ("cad_lin",          "cad_vec_lin",      "COMPRIMENTO",  "acrescenta o comprimento do fragmento"),
    ("cad_vec_lin",      "cad_vec_lin_frag", "NORMALIZAÇÃO", "padronização por paciente → por fragmento"),
    ("cad_vec_lin_frag", "cad_vec_mlpdir",   "CLASSIFICADOR","troca a regressão logística por uma rede"),
    ("cad_vec_mlpdir",   "cad_vec_mlp",      "PROJEÇÃO",     "acrescenta uma projeção DEPOIS da média"),
    ("cad_vec_mlp",      "cad_mean",         "ORDEM",        "move a mesma projeção para ANTES da média"),
    ("cad_mean",         "cad_att",          "AGREGADOR",    "troca a média por atenção aprendida"),
]

print("\n" + "=" * 100)
print("A ESCADA PRINCIPAL — SEIS DEGRAUS SOBRE A REPRESENTAÇÃO DO FOUNDATION MODEL")
print("=" * 100)
t0 = time.time()
degraus = monta_escada(ESCADA)
soma, extremos, discrep = verifica_telescopio(degraus, "cad_lin", "cad_att", "principal")
print(f"  (bootstrap: {time.time()-t0:.0f} s)")
pior = imprime_escada(degraus, "principal")
print(f"\n  O bootstrap reamostra os mesmos {N_DEV} pacientes {CFG['N_BOOTSTRAP']} vezes.")
print(f"  'Sinal' conta em quantas das {degraus[0]['n_comp']} comparações "
      f"(semente × dobra) o degrau foi negativo.")
print(f"\n  VERIFICAÇÃO DE REGRESSÃO: estes seis valores devem reproduzir a execução")
print(f"  anterior. Os braços acrescentados nesta revisão recebem sementes próprias")
print(f"  e não perturbam os existentes. Divergência indica defeito, não descoberta.")

# -----------------------------------------------------------------------------
# Análise de sensibilidade à otimização.
#
# Um degrau pode ficar grande porque o modelo de destino é pior, ou porque ele
# não conseguiu treinar. Esta tabela tenta separar as duas.
# -----------------------------------------------------------------------------
N_EXEC_ABAIXO = int(sum(1 for r in REPLICAS for b in NEURAIS
                        for a in r[b]["pico_suave"] if a < CFG["LIMIAR_PARADA"]))
N_EXEC_TOTAL = len(NEURAIS) * len(CFG["SEMENTES"]) * CFG["N_DOBRAS"]
N_COMP_AFETADAS = sum(d["n_comp"] - d["n_ok"] for d in degraus)

print("\n" + "-" * 100)
print("ANÁLISE DE SENSIBILIDADE À OTIMIZAÇÃO")
print(f"Marca as execuções cuja AUROC suavizada no conjunto de parada superou "
      f"{CFG['LIMIAR_PARADA']:.2f}.")
print("-" * 100)
print(f"  {'Degrau':<14s} {'Atingiram':>11s} {'Δ (atingiram)':>22s} "
      f"{'Δ (não atingiram)':>22s} {'Δ (todas)':>11s}")
print("  " + "-" * 86)
for d in degraus:
    c = (f"{d['delta_ok']:+.3f} ({d['neg_ok']}/{d['n_ok']} neg)"
         if np.isfinite(d["delta_ok"]) else "—")
    nf = d["n_comp"] - d["n_ok"]
    f_ = (f"{d['delta_falha']:+.3f} ({d['neg_falha']}/{nf} neg)"
          if np.isfinite(d["delta_falha"]) else "—")
    print(f"  {d['etiqueta']:<14s} {d['n_ok']:>6d}/{d['n_comp']:<3d} "
          f"{c:>22s} {f_:>22s} {d['delta']:+11.3f}")
print(f"\n  Execuções abaixo do limiar : {N_EXEC_ABAIXO} de {N_EXEC_TOTAL}")
if N_COMP_AFETADAS == 0:
    print(f"  TODAS as execuções atingiram o limiar. A tentativa de separar falha de")
    print(f"  otimização de efeito arquitetural NÃO foi conclusiva nesta execução:")
    print(f"  vale apenas o resultado sobre as {degraus[0]['n_comp']} comparações.")

# -----------------------------------------------------------------------------
# O mesmo contraste noutra representação. Se um degrau se repete com o mesmo
# sinal em 4-mers, ele não é propriedade do Foundation Model.
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("OS MESMOS DEGRAUS, MEDIDOS EM 4-MERS")
print("-" * 100)
replicacao = []
for etiq, oc, dc, ok_, dk in [
        ("ORDEM", "cad_vec_mlp", "cad_mean", "kmer_vec_mlp", "kmer_mean"),
        ("AGREGADOR", "cad_mean", "cad_att", "kmer_mean", "kmer_att")]:
    pc = np.array([r[dc]["media"] - r[oc]["media"] for r in REPLICAS])
    pk = np.array([r[dk]["media"] - r[ok_]["media"] for r in REPLICAS])
    dif_k = np.concatenate([r[dk]["auc_dobras"] - r[ok_]["auc_dobras"] for r in REPLICAS])
    bk, bklo, bkhi = bootstrap_delta_auc(Y_DEV, OOF_ENS[ok_], OOF_ENS[dk],
                                         CFG["N_BOOTSTRAP"], SEMENTE_MESTRA)
    replicacao.append(dict(degrau=etiq, caduceus=float(pc.mean()), dp_cad=float(pc.std(ddof=1)),
                           kmers=float(pk.mean()), dp_kmer=float(pk.std(ddof=1)),
                           boot_kmer=bk, boot_kmer_inf=bklo, boot_kmer_sup=bkhi,
                           neg_kmer=int((dif_k < 0).sum()), n_kmer=len(dif_k),
                           mesmo_sinal=bool(np.sign(pc.mean()) == np.sign(pk.mean()))))
    print(f"  {etiq:<11s} Caduceus {pc.mean():+.3f} ± {pc.std(ddof=1):.3f}   "
          f"4-mers {pk.mean():+.3f} ± {pk.std(ddof=1):.3f} "
          f"[IC {bklo:+.3f}, {bkhi:+.3f}]  ({int((dif_k < 0).sum())}/{len(dif_k)} neg)   "
          f"{'mesmo sinal' if np.sign(pc.mean())==np.sign(pk.mean()) else 'SINAIS OPOSTOS'}")

# --- controles de otimização -------------------------------------------------
tab_clip = pd.DataFrame([dict(
    Braco=nome(b),
    Taxa_recorte_gradiente=sum(r[b]["cortes"] for r in REPLICAS) /
                           max(sum(r[b]["passos"] for r in REPLICAS), 1),
    Epocas_medias=float(np.mean([e for r in REPLICAS for e in r[b]["epocas"]])),
    Pico_parada_bruto=float(np.median([a for r in REPLICAS for a in r[b]["pico_cru"]])),
    Pico_parada_suave=float(np.median([a for r in REPLICAS for a in r[b]["pico_suave"]])),
    Abaixo_do_limiar=int(sum(1 for r in REPLICAS for a in r[b]["pico_suave"]
                             if a < CFG["LIMIAR_PARADA"])))
    for b in NEURAIS]).sort_values("Abaixo_do_limiar", ascending=False)
TAXA_CLIP = float(tab_clip["Taxa_recorte_gradiente"].max())
INFLACAO_PICO = float((tab_clip["Pico_parada_bruto"] - tab_clip["Pico_parada_suave"]).mean())
print("\n" + "-" * 100)
print("CONTROLES DE OTIMIZAÇÃO")
print("A diferença entre o pico bruto e o suavizado é a inflação por tomar o máximo")
print("de muitas avaliações ruidosas. A triagem usa o suavizado.")
print("-" * 100)
print(tab_clip.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\n  Inflação média do pico bruto sobre o suavizado: {INFLACAO_PICO:+.3f} de AUROC.")

# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 100)
print("  • A escada depende do caminho: outra ordem dos degraus daria outras fatias")
print("    com a mesma soma. A identidade telescópica verifica o cálculo, não a causa.")
print(f"  • As {degraus[0]['n_comp']} comparações não são independentes — os conjuntos de treino se")
print("    sobrepõem e os pacientes se repetem. A contagem de sinais é descritiva.")
print("  • NORMALIZAÇÃO muda a referência de padronização E a força efetiva da")
print("    penalização L2, juntas. Não isola escala pura. Mas a dependência de escala")
print("    da penalização é propriedade real de qualquer classificador regularizado.")
print("  • O limiar de parada não mede convergência do otimizador. Tomar o máximo de")
print(f"    dezenas de avaliações num conjunto de ~{len(_d0['parada'])} pacientes cruza qualquer limiar")
print("    fixo com frequência apreciável. A suavização reduz a inflação, não a elimina.")
print("  • O degrau ORDEM mede a consequência de ponta a ponta da escolha")
print("    arquitetural, incluindo o efeito sobre a otimização. Os dois braços são")
print("    treinados separadamente, isso é consequência da escolha, não confundimento.")

print("\n" + "=" * 100); print("FIM DA CÉLULA 4"); print("=" * 100)

# =============================================================================
# ===== FIM CÉLULA 4 ==========================================================
# =============================================================================

CÉLULA 4 — A ESCADA DE SEIS DEGRAUS, REPETIDA EM CINCO SEMENTES

  Braços neurais  : 11
  Braços lineares : 8 (mais o resíduo, se computável)

  Embeddings: float16, 1.82 GB cada · GPU precisa 3.83 GB de 14.46 GB livres
  Pico de RAM do host na subida: 0.119 GB de 29.22 GB livres


/tmp/ipykernel_23/2870752395.py:68: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  t[i:i+lote] = torch.from_numpy(np.ascontiguousarray(mm[i:i+lote])).to(DISPOSITIVO)


  Bolsas residentes na GPU.

RÉPLICA 1 de 5 — semente 1807
    Caduceus + Comprimento (Projeção → Atenção MIL)                    0.751  ép 19  (13s)
    Caduceus + Comprimento (Projeção → Média → MLP)                    0.743  ép 29  (9s)
    Caduceus + Comprimento (Média → Projeção → MLP)                    0.813  ép 21  (2s)
    Caduceus + Comprimento (Média → MLP)                               0.833  ép 47  (3s)
    4-mers + Comprimento (Projeção → Atenção MIL)                      0.797  ép 12  (6s)
    4-mers + Comprimento (Projeção → Média → MLP)                      0.776  ép 28  (8s)
    4-mers + Comprimento (Média → Projeção → MLP)                      0.816  ép 47  (3s)
    Somente Comprimento (Atenção MIL) [canal único]                    0.668  ép  9  (5s)
    Cobertura DELFI-like, painel inteiro (MLP)                         0.838  ép  6  (1s)
    Cobertura DELFI-like, promotora, todos os fragmentos (MLP)         0.829  ép  4  (1s)
    Cobertura DELFI-like, promotora, os 

In [5]:
# =============================================================================
# ===== CÉLULA 5 — A ESCADA DA COBERTURA: REGIÃO E PROFUNDIDADE ===============
# =============================================================================
#
# POR QUE ESTA ESCADA EXISTE
#
# A cobertura em janelas de megabases enxerga todos os fragmentos do painel. As
# representações de resolução de base enxergam 5.000. Comparar as duas confunde
# duas coisas: a escala genômica e a quantidade de moléculas.
#
# O conjunto de features contém um terceiro arquivo, gerado exatamente para
# desfazer essa confusão: a mesma cobertura, nas mesmas janelas, calculada
# sobre os mesmos 5.000 fragmentos da bolsa.
#
#   Cobertura no painel inteiro, todos os fragmentos
#     │ REGIÃO         restringe as janelas às regiões promotoras
#     │ PROFUNDIDADE   restringe aos mesmos 5.000 fragmentos da bolsa
#   Cobertura promotora, os mesmos 5.000
#
# O degrau PROFUNDIDADE é o mais limpo de todo o estudo: mesma região, mesmas
# janelas, mesmo classificador, mesmas partições. Só o número de moléculas muda.
#
# O degrau REGIÃO não é igualmente limpo. Restringir a região altera, por
# consequência, quais janelas têm cobertura suficiente para entrar no vetor.
# Isso é declarado, não corrigido: a alternativa seria comparar janelas que não
# existem nas duas condições.
# =============================================================================

print("=" * 100)
print("CÉLULA 5 — A ESCADA DA COBERTURA: REGIÃO E PROFUNDIDADE")
print("=" * 100)

ESCADA_COBERTURA_MLP = [
    ("delfi",          "delfi_prom_all",     "REGIÃO",
     "restringe as janelas às regiões promotoras"),
    ("delfi_prom_all", "delfi_prom_matched", "PROFUNDIDADE",
     "restringe aos mesmos 5.000 fragmentos da bolsa"),
]
ESCADA_COBERTURA_LOG = [
    ("rep_delfi",          "rep_delfi_prom_all",     "REGIÃO",
     "restringe as janelas às regiões promotoras"),
    ("rep_delfi_prom_all", "rep_delfi_prom_matched", "PROFUNDIDADE",
     "restringe aos mesmos 5.000 fragmentos da bolsa"),
]

print("\n" + "-" * 100)
print("FAMÍLIA REDE (MLP)")
print("-" * 100)
degraus_cob_mlp = monta_escada(ESCADA_COBERTURA_MLP)
soma_cob_mlp, ext_cob_mlp, disc_cob_mlp = verifica_telescopio(
    degraus_cob_mlp, "delfi", "delfi_prom_matched", "da cobertura (rede)")
imprime_escada(degraus_cob_mlp, "cobertura MLP")

print("\n" + "-" * 100)
print("FAMÍLIA REGRESSÃO LOGÍSTICA")
print("-" * 100)
degraus_cob_log = monta_escada(ESCADA_COBERTURA_LOG)
soma_cob_log, ext_cob_log, disc_cob_log = verifica_telescopio(
    degraus_cob_log, "rep_delfi", "rep_delfi_prom_matched", "da cobertura (logística)")
imprime_escada(degraus_cob_log, "cobertura logística")

# -----------------------------------------------------------------------------
# O MESMO CONTRASTE NA COORTE EXTERNA
#
# A escada acima é interna. A pergunta que motiva tudo isto é sobre
# transferência: a cobertura sobrevive à mudança de coorte porque enxerga mais
# moléculas, ou porque a escala de megabase é robusta? Só a coorte externa
# responde.
# -----------------------------------------------------------------------------
cond_ext_pre = META["condicao"].values[IDX_EXT]
mask_hcc_pre = (cond_ext_pre == HCC)
sel_pre = mask_hcc_pre | np.isin(cond_ext_pre, list(SAUDAVEL))
y_ext_pre = mask_hcc_pre[sel_pre].astype(int)

print("\n" + "-" * 100)
print("A MESMA ESCADA, MEDIDA NA COORTE EXTERNA")
print("A coorte externa não é homóloga: outro tipo tumoral, outra população,")
print("outro protocolo. O valor aqui é o contraste entre os três, não o nível.")
print("-" * 100)
escada_externa = []
for familia, espec in (("rede", ESCADA_COBERTURA_MLP),
                       ("logística", ESCADA_COBERTURA_LOG)):
    print(f"\n  família {familia}")
    print(f"  {'braço':<58s} {'interna':>9s} {'externa':>9s}")
    braços_fam = [espec[0][0]] + [d[1] for d in espec]
    aucs_int, aucs_ext = {}, {}
    for b in braços_fam:
        a_int = float(np.mean([r[b]["media"] for r in REPLICAS]))
        e_med = np.mean([r[b]["externo"] for r in REPLICAS], axis=0)
        a_ext = float(roc_auc_score(y_ext_pre, e_med[sel_pre]))
        aucs_int[b], aucs_ext[b] = a_int, a_ext
        print(f"  {nome(b):<58s} {a_int:>9.3f} {a_ext:>9.3f}")
    for orig, dest, etiq, _ in espec:
        escada_externa.append(dict(
            familia=familia, etiqueta=etiq,
            delta_interna=aucs_int[dest] - aucs_int[orig],
            delta_externa=aucs_ext[dest] - aucs_ext[orig]))
        print(f"    degrau {etiq:<14s} interna {aucs_int[dest]-aucs_int[orig]:+.3f}   "
              f"externa {aucs_ext[dest]-aucs_ext[orig]:+.3f}")

# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("LEITURA")
print("-" * 100)
_d_reg = [d for d in degraus_cob_mlp if d["etiqueta"] == "REGIÃO"][0]
_d_prof = [d for d in degraus_cob_mlp if d["etiqueta"] == "PROFUNDIDADE"][0]
print(f"  Restringir a região custa {_d_reg['delta']:+.3f} de AUROC interna.")
print(f"  Restringir a profundidade custa {_d_prof['delta']:+.3f}.")
_raz = abs(_d_prof['delta']) / max(abs(_d_reg['delta']), 1e-9)
print(f"  A profundidade pesa {_raz:.1f}x mais que a região.")
print(f"\n  Com {DELFI_PROM_ALL.shape[1]//3} janelas e {CFG['N_FRAGMENTOS']:,} fragmentos, cada janela")
print(f"  recebe {CFG['N_FRAGMENTOS']/(DELFI_PROM_ALL.shape[1]//3):.1f} moléculas em média, divididas ainda entre curtas e")
print(f"  longas. Uma razão calculada sobre contagens desse porte é dominada por")
print(f"  ruído de amostragem. É o mesmo mecanismo que limita o perfil de motivos.")

print("\n" + "-" * 100)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 100)
print("  • O degrau REGIÃO altera a região E o conjunto de janelas com cobertura")
print("    suficiente. Não isola a região sozinha. O degrau PROFUNDIDADE isola.")
print("  • A cobertura completa não tem contagem de fragmentos gravada por")
print("    paciente, então a razão exata de profundidade entre os dois extremos")
print("    não é calculável aqui.")
print("  • A escada externa não tem os dois estimadores: com 90 casos e 32")
print("    controles, o bootstrap pareado teria largura maior que o efeito.")

print("\n" + "=" * 100); print("FIM DA CÉLULA 5"); print("=" * 100)

# =============================================================================
# ===== FIM CÉLULA 5 ==========================================================
# =============================================================================

CÉLULA 5 — A ESCADA DA COBERTURA: REGIÃO E PROFUNDIDADE

----------------------------------------------------------------------------------------------------
FAMÍLIA REDE (MLP)
----------------------------------------------------------------------------------------------------
  Soma dos degraus            : -0.2208
  Diferença entre os extremos : -0.2208
  Discrepância                : 0.00e+00
  Aprovado: a decomposição é aritmeticamente exata.

  Degrau         O que muda                                              Δ     ±dp        Sinal
  ----------------------------------------------------------------------------------------------
  REGIÃO         restringe as janelas às regiões promotoras         -0.006   0.011   14/25 neg
  PROFUNDIDADE   restringe aos mesmos 5.000 fragmentos da bolsa     -0.215   0.029   25/25 neg  ← MAIOR PERDA

  ----------------------------------------------------------------------------------------------
  Degrau           média das diferenças por dobra   

In [6]:
# =============================================================================
# ===== CÉLULA 6 — ENCODER, GENERALIZAÇÃO E COORTE ============================
# =============================================================================

print("=" * 100)
print("CÉLULA 6 — ENCODER, GENERALIZAÇÃO E COORTE")
print("=" * 100)

RES = {}
for k in CHAVES:
    m = np.array([r[k]["media"] for r in REPLICAS])
    auc_ens = float(roc_auc_score(Y_DEV, OOF_ENS[k]))
    # Concordância entre réplicas: correlação de Spearman média par a par entre
    # os escores das cinco réplicas nos mesmos pacientes. Mede se elas ORDENAM
    # os pacientes da mesma forma.
    oofs = [r[k]["oof"] for r in REPLICAS]
    pares = [spearmanr(oofs[i], oofs[j]).correlation
             for i in range(len(oofs)) for j in range(i+1, len(oofs))]
    RES[k] = dict(media=float(m.mean()), dp_sementes=float(m.std(ddof=1)),
                  oof=OOF_ENS[k], auc_oof_ens=auc_ens,
                  ganho_ensemble=auc_ens - float(m.mean()),
                  concordancia=float(np.mean(pares)),
                  externo=np.mean([r[k]["externo"] for r in REPLICAS], axis=0),
                  externo_por_semente=[r[k]["externo"] for r in REPLICAS])

tab_int = pd.DataFrame([dict(Braco=nome(k), AUROC=RES[k]["media"],
                             DP_entre_sementes=RES[k]["dp_sementes"],
                             AUROC_ensemble=RES[k]["auc_oof_ens"],
                             Ganho_do_ensemble=RES[k]["ganho_ensemble"],
                             Concordancia_entre_replicas=RES[k]["concordancia"])
                        for k in CHAVES]).sort_values("AUROC", ascending=False)
print("\n" + "-" * 100)
print("DESEMPENHO INTERNO (BALTIMORE) — 5 SEMENTES × 5 DOBRAS")
print("'AUROC' é a média das 25 dobras. 'AUROC_ensemble' usa o escore médio das cinco")
print("réplicas, é o modelo que de fato avaliamos fora. 'Ganho_do_ensemble' mede o")
print("efeito de AGREGAR escores; 'Concordancia' mede se as réplicas ORDENAM os")
print("pacientes da mesma forma. São coisas diferentes.")
print("-" * 100)
print(tab_int.to_string(index=False, formatters={
    "AUROC": lambda v: f"{v:.3f}", "DP_entre_sementes": lambda v: f"{v:.3f}",
    "AUROC_ensemble": lambda v: f"{v:.3f}", "Ganho_do_ensemble": lambda v: f"{v:+.3f}",
    "Concordancia_entre_replicas": lambda v: f"{v:.3f}"}))

_at = [k for k in CHAVES if AGREGACAO.get(k) == "atencao"]
_md = [k for k in CHAVES if AGREGACAO.get(k) == "media_apos_projecao"]
if _at and _md:
    c_at = float(np.mean([RES[k]["concordancia"] for k in _at]))
    c_md = float(np.mean([RES[k]["concordancia"] for k in _md]))
    g_at = float(np.mean([RES[k]["ganho_ensemble"] for k in _at]))
    g_md = float(np.mean([RES[k]["ganho_ensemble"] for k in _md]))
    print(f"\n  {'':28s} {'concordância':>13s} {'ganho ensemble':>16s}")
    print(f"  {'braços com ATENÇÃO':<28s} {c_at:>13.3f} {g_at:>+16.3f}")
    print(f"  {'braços com MÉDIA':<28s} {c_md:>13.3f} {g_md:>+16.3f}")

# -----------------------------------------------------------------------------
# CONCORDÂNCIA ANTES E DEPOIS DO DEGRAU ORDEM
#
# A comparação certa é entre a arquitetura imediatamente ANTES e a imediatamente
# DEPOIS de mover a projeção. Comparar atenção com média-após-projeção compara
# dois braços que já estão do mesmo lado do degrau.
# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("CONCORDÂNCIA ENTRE RÉPLICAS, ANTES E DEPOIS DO DEGRAU ORDEM")
print("-" * 100)
PARES_ORDEM = [("Caduceus", "cad_vec_mlp", "cad_mean"),
               ("4-mers",   "kmer_vec_mlp", "kmer_mean")]
for etiq, antes, depois in PARES_ORDEM:
    print(f"  {etiq:<10s} antes {RES[antes]['concordancia']:.3f}   "
          f"depois {RES[depois]['concordancia']:.3f}   "
          f"queda {RES[depois]['concordancia']-RES[antes]['concordancia']:+.3f}")

FALHAS_POR_LADO = {"projeta fragmento a fragmento": 0, "agrega antes da não linearidade": 0}
for r in REPLICAS:
    for b in NEURAIS:
        antes_da_nao_lin = AGREGACAO.get(b) in ("media_antes_projecao", "media_sem_projecao")
        for a in r[b]["pico_suave"]:
            if a < CFG["LIMIAR_PARADA"]:
                chave = ("agrega antes da não linearidade" if antes_da_nao_lin
                         else "projeta fragmento a fragmento")
                FALHAS_POR_LADO[chave] += 1
print(f"\n  Execuções abaixo do limiar, por lado do degrau:")
for k, v in FALHAS_POR_LADO.items():
    print(f"    {k:<38s} {v:>3d}")

# --- O QUE O ENCODER ACRESCENTA ---------------------------------------------
print("\n" + "=" * 100)
print("O QUE O ENCODER ACRESCENTA")
print("=" * 100)
print(f"  Reconstrução do vetor do Caduceus:")
print(f"    só composição de 4-mers   : R² = {R2_KMER:.3f}")
print(f"    só ocupação de promotores : R² = {R2_OCUP:.3f}")
print(f"    as duas juntas            : R² = {R2_RESID:.3f}")
if RESID_OK:
    ra = RES["rep_cad_resid"]["media"]; ca = RES["cad_lin"]["media"]
    rk = RES["rep_kmer"]["media"]
    n_exc = int(np.sum(NULOS_RESID >= ra)); n_perm = len(NULOS_RESID)
    P_RESID = float((1 + n_exc) / (1 + n_perm))
    p_lo = float(beta.ppf(0.025, n_exc, n_perm - n_exc + 1)) if n_exc > 0 else 0.0
    p_hi = float(beta.ppf(0.975, n_exc + 1, n_perm - n_exc)) if n_exc < n_perm else 1.0
    gan, gan_lo, gan_hi = bootstrap_delta_auc(
        Y_DEV, OOF_ENS["rep_kmer"], OOF_ENS["cad_lin"], CFG["N_BOOTSTRAP"], SEMENTE_MESTRA)
    print(f"\n  Poder discriminante (AUROC no desenvolvimento):")
    print(f"    Vetor completo do Caduceus   : {ca:.3f}")
    print(f"    Composição de 4-mers sozinha : {rk:.3f}")
    print(f"    Apenas o resíduo             : {ra:.3f}")
    print(f"    Nulo do resíduo, percentil 95: {PERC95_RESID:.3f}")
    print(f"    p empírico : {P_RESID:.4f}   ({n_exc} de {n_perm} permutações ≥ observado)")
    print(f"    IC 95% de Monte Carlo do p : [{p_lo:.4f}, {p_hi:.4f}]")
    print(f"\n  Ganho do encoder sobre contar 4-mers : {gan:+.3f} "
          f"[IC 95% {gan_lo:+.3f}, {gan_hi:+.3f}]")
    if P_RESID <= 0.05:
        print(f"\n  LEITURA: há sinal residual sob esta reconstrução linear. Quase toda")
        print(f"  a variância do encoder é composição de bases, mas a fração que não")
        print(f"  é carrega sinal diagnóstico.")
    else:
        print(f"\n  LEITURA: sob esta reconstrução o resíduo não se distingue do nulo.")
else:
    P_RESID = p_lo = p_hi = gan = gan_lo = gan_hi = float("nan")

# --- GENERALIZAÇÃO PARA A COORTE NÃO HOMÓLOGA -------------------------------
cond_ext = META["condicao"].values[IDX_EXT]
mask_hcc = (cond_ext == HCC)
sel = mask_hcc | np.isin(cond_ext, list(SAUDAVEL))
y_ext = mask_hcc[sel].astype(int)

def ic(y, p, n_boot):
    """Bootstrap dos pacientes, com os modelos fixos."""
    rng = np.random.RandomState(SEMENTE_MESTRA); v = []
    for _ in range(n_boot):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) > 1:
            v.append(roc_auc_score(y[i], p[i]))
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if v else (np.nan, np.nan)

print("\n" + "#" * 100)
print("#  GENERALIZAÇÃO PARA UMA COORTE NÃO HOMÓLOGA")
print("#  Carcinoma hepatocelular contra saudáveis, num tipo tumoral ausente do")
print("#  treino e numa população com outra ascendência, centro e protocolo.")
print("#" * 100)

linhas = []
for k in CHAVES:
    por_sem = np.array([roc_auc_score(y_ext, e[sel]) for e in RES[k]["externo_por_semente"]])
    a = roc_auc_score(y_ext, RES[k]["externo"][sel])
    lo, hi = ic(y_ext, RES[k]["externo"][sel], CFG["N_BOOTSTRAP"])
    RES[k]["auc_ext"] = a
    linhas.append(dict(Braco=nome(k), AUROC_externa=a, DP_sementes=float(por_sem.std(ddof=1)),
                       IC_inf=lo, IC_sup=hi, AUROC_interna=RES[k]["media"],
                       Variacao=a - RES[k]["media"]))
tab_ext = pd.DataFrame(linhas).sort_values("AUROC_externa", ascending=False)
print(f"\n{int(y_ext.sum())} casos / {int((1-y_ext).sum())} controles")
print("-" * 100)
print(tab_ext.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\n" + "-" * 100)
print(f"PONTO DE OPERAÇÃO EXPLORATÓRIO — limiar fixado em "
      f"{100*CFG['ESPECIFICIDADE_ALVO']:.0f}% de especificidade no desenvolvimento")
print("-" * 100)
linhas_op = []
for k in CHAVES:
    lim = np.percentile(RES[k]["oof"][Y_DEV == 0], 100 * CFG["ESPECIFICIDADE_ALVO"])
    p = RES[k]["externo"][sel]
    linhas_op.append(dict(Braco=nome(k),
        Especificidade=float(((p < lim) & (y_ext == 0)).sum() / max((y_ext == 0).sum(), 1)),
        Sensibilidade=float(((p >= lim) & (y_ext == 1)).sum() / max((y_ext == 1).sum(), 1))))
tab_op = pd.DataFrame(linhas_op).sort_values("Sensibilidade", ascending=False)
print(tab_op.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("  A especificidade mostrada é a obtida na coorte externa, não a pedida.")

# --- O QUE A ATENÇÃO FAZ -----------------------------------------------------
print("\n" + "-" * 100)
print("O QUE A ATENÇÃO FAZ")
print("A justificativa usual do Attention-MIL é que ele encontra as instâncias")
print("informativas. 'N efetivo' = 1 / soma dos pesos ao quadrado: a quantos")
print(f"fragmentos de peso igual a distribuição equivale. Uniforme seria {CFG['N_FRAGMENTOS']}.")
print("-" * 100)
linhas_at = []
for b in NEURAIS:
    todos = [w for r in REPLICAS for w in r[b]["pesos"]]
    if not todos: continue
    n_ef = [float(np.median(1.0 / (w ** 2).sum(axis=1))) for w in todos]
    rho  = [float(np.nanmedian([spearmanr(w[i], COMP_ORD[N_DEV:][i]).correlation
                                for i in range(min(40, len(w)))])) for w in todos]
    linhas_at.append(dict(Braco=nome(b), N_efetivo=float(np.median(n_ef)),
                          Fracao_da_bolsa=float(np.median(n_ef))/CFG["N_FRAGMENTOS"],
                          Correlacao_com_comprimento=float(np.median(rho))))
tab_at = pd.DataFrame(linhas_at)
print(tab_at.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# --- AS COORTES SÃO DISTINGUÍVEIS ENTRE PESSOAS SAUDÁVEIS? -------------------
print("\n" + "=" * 100)
print("QUANTO DE CADA REPRESENTAÇÃO É ASSINATURA DE COORTE?")
print("Só indivíduos SAUDÁVEIS das duas coortes. O rótulo é a coorte, não a doença.")
print("Com quantas direções principais se separa um saudável de Baltimore de um de")
print("Hong Kong? Menos componentes = diferença de coorte mais dominante.")
print("=" * 100)
sd_ = np.where((META["estudo"].values == ESTUDO_DEV) & META["condicao"].isin(SAUDAVEL).values)[0]
se_ = np.where((META["estudo"].values == ESTUDO_EXT) & META["condicao"].isin(SAUDAVEL).values)[0]
loc = POS[np.concatenate([sd_, se_])]
y_co = np.r_[np.zeros(len(sd_)), np.ones(len(se_))].astype(int)
print(f"  {len(sd_)} saudáveis de Baltimore contra {len(se_)} de Hong Kong "
      f"(classes desbalanceadas: a regressão usa pesos compensados)")

KL = sorted(CFG["K_COMPONENTES_PCA"]); KMAX = max(KL)
skfc = StratifiedKFold(n_splits=CFG["N_DOBRAS"], shuffle=True, random_state=SEMENTE_MESTRA)
linhas_co = []
for nm, Xr in [("Cobertura DELFI-like, painel", V_DELFI[loc]),
               ("Cobertura DELFI-like, promotora", V_DELFI_PROM_ALL[loc]),
               ("Cobertura promotora, os mesmos 5.000", V_DELFI_PROM_MATCHED[loc]),
               ("Ocupação de Promotores", V_OCUP[loc]),
               ("Perfil de End-Motif", V_MOTIF[loc]),
               ("Composição de 4-mers", V_KMER[loc]),
               ("Embedding do Caduceus", V_CAD[loc])]:
    pr = {k: np.zeros(len(y_co)) for k in KL}
    for tr, te in skfc.split(Xr, y_co):
        e = StandardScaler().fit(Xr[tr])
        Xt, Xv = e.transform(Xr[tr]), e.transform(Xr[te])
        kd = min(KMAX, len(tr)-1, Xr.shape[1])
        # Uma única decomposição, fatiada: os subespaços ficam aninhados.
        pc = PCA(n_components=kd, svd_solver="randomized",
                 random_state=SEMENTE_MESTRA).fit(Xt)
        Zt, Zv = pc.transform(Xt), pc.transform(Xv)
        for k in KL:
            kk = min(k, kd)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
                    Zt[:, :kk], y_co[tr])
            pr[k][te] = lr.predict_proba(Zv[:, :kk])[:, 1]
    linha = dict(Representacao=nm, Dimensoes=Xr.shape[1]); k95 = None
    for k in KL:
        a = roc_auc_score(y_co, pr[k]); linha[f"k={k}"] = a
        if k95 is None and a >= 0.95: k95 = k
    linha["k_para_0.95"] = k95 if k95 else f">{KMAX}"
    linhas_co.append(linha)
tab_co = pd.DataFrame(linhas_co)
print()
print(tab_co.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# -----------------------------------------------------------------------------
print("\n" + "-" * 100)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 100)
print("  • A queda na coorte externa confunde TRÊS causas: tipo tumoral, ascendência")
print("    e protocolo. A quarta — amplitude de amostragem — deixou de ser")
print("    confundidor e foi medida na Célula 5.")
print(f"  • Os máximos de AUROC foram escolhidos depois de ver os dados. Com "
      f"{len(CHAVES)} braços,")
print("    o vencedor carrega viés de seleção; as sementes descrevem a instabilidade,")
print("    não a removem.")
print("  • 'N efetivo' é propriedade da DISTRIBUIÇÃO DE PESOS, não uma contagem de")
print("    fragmentos informativos. São conceitos diferentes.")
print("  • O teste de coorte NÃO estabelece causa: as coortes diferem em ascendência,")
print("    centro, protocolo e provavelmente idade, sem metadados para separá-los.")
print("  • O ganho do encoder sobre 4-mers é um contraste entre representações, não")
print("    a identificação do pré-treino como causa desse ganho.")

print("\n" + "=" * 100); print("FIM DA CÉLULA 6"); print("=" * 100)

# =============================================================================
# ===== FIM CÉLULA 6 ==========================================================
# =============================================================================

CÉLULA 6 — ENCODER, GENERALIZAÇÃO E COORTE

----------------------------------------------------------------------------------------------------
DESEMPENHO INTERNO (BALTIMORE) — 5 SEMENTES × 5 DOBRAS
'AUROC' é a média das 25 dobras. 'AUROC_ensemble' usa o escore médio das cinco
réplicas, é o modelo que de fato avaliamos fora. 'Ganho_do_ensemble' mede o
efeito de AGREGAR escores; 'Concordancia' mede se as réplicas ORDENAM os
pacientes da mesma forma. São coisas diferentes.
----------------------------------------------------------------------------------------------------
                                                           Braco AUROC DP_entre_sementes AUROC_ensemble Ganho_do_ensemble Concordancia_entre_replicas
                 Caduceus + Comprimento (Média → Projeção → MLP) 0.857             0.009          0.865            +0.008                       0.925
                            Caduceus + Comprimento (Média → MLP) 0.854             0.008          0.857            +0.003 

In [7]:
# =============================================================================
# ===== CÉLULA 7 — FIGURAS, EXPORTAÇÃO E SÍNTESE ==============================
# =============================================================================

print("=" * 100); print("CÉLULA 7 — FIGURAS, EXPORTAÇÃO E SÍNTESE"); print("=" * 100)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.titlesize": 10, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.bbox": "tight", "font.family": "DejaVu Sans"})
C_CAD, C_KMER, C_DELFI = "#2166AC", "#B2182B", "#1B7837"
C_MOTIF = "#E08214"
C_SOBE, C_DESCE = "#4393C3", "#D6604D"
inv = {nome(k): k for k in CHAVES}

# ============================ FIGURA 1 =======================================
fig = plt.figure(figsize=(15.0, 6.6))
gs = fig.add_gridspec(1, 2, width_ratios=[2.1, 1], wspace=0.28)
ax = fig.add_subplot(gs[0, 0])

base = RES["cad_lin"]["media"]
acum, pos, alt, bas, cor, eti, erro = base, [], [], [], [], [], []
for i, d in enumerate(degraus):
    pos.append(i); alt.append(d["delta"]); erro.append(d["dp_sementes"])
    bas.append(acum if d["delta"] > 0 else acum + d["delta"])
    cor.append(C_SOBE if d["delta"] > 0 else C_DESCE)
    eti.append(d["etiqueta"]); acum += d["delta"]

ax.bar([-1], [base], color="#999999", width=0.62)
ax.bar(pos, np.abs(alt), bottom=bas, color=cor, width=0.62,
       yerr=erro, capsize=3, error_kw=dict(lw=1.0, ecolor="#333333"))
ax.bar([len(degraus)], [acum], color="#333333", width=0.62)
topos = [b + abs(h) + e for b, h, e in zip(bas, alt, erro)] + [base, acum]
pisos = list(bas) + [base, acum]
folga = max(0.04, 0.22 * (max(topos) - min(pisos)))
ax.set_ylim(min(pisos) - folga * 1.15, max(topos) + folga)
for p_, h, b_, e in zip(pos, alt, bas, erro):
    ax.text(p_, b_ + abs(h) + e + 0.006, f"{h:+.3f}", ha="center", va="bottom",
            fontsize=8.5, fontweight="bold")
ax.text(-1, base + 0.006, f"{base:.3f}", ha="center", va="bottom",
        fontsize=8.5, fontweight="bold")
ax.text(len(degraus), acum + 0.006, f"{acum:.3f}", ha="center", va="bottom",
        fontsize=8.5, fontweight="bold")
ax.set_xticks([-1] + pos + [len(degraus)])
ax.set_xticklabels([CURTO["cad_lin"]] + eti + [CURTO["cad_att"]], rotation=34,
                   ha="right", fontsize=8)
ax.set_ylabel("AUROC no desenvolvimento interno")
ax.set_title("A · A escada: cada barra muda uma única coisa", loc="left")
i_p = int(np.argmin(alt))
ax.annotate(f"maior perda\n({eti[i_p]})", xy=(i_p, bas[i_p]),
            xytext=(i_p - 0.60, min(pisos) - folga * 0.85), fontsize=8.5,
            fontweight="bold", color=C_KMER,
            arrowprops=dict(arrowstyle="->", color=C_KMER, lw=1.2))
ax.set_xlabel(f"Soma dos degraus {soma:+.4f} = diferença entre extremos {extremos:+.4f} "
              f"(identidade aritmética)\nbarras = média das diferenças por dobra · "
              f"erro = desvio entre {len(CFG['SEMENTES'])} sementes",
              fontsize=8, style="italic", labelpad=14)

ax = fig.add_subplot(gs[0, 1])
w, xs = 0.35, np.arange(len(replicacao))
ax.bar(xs - w/2, [r["caduceus"] for r in replicacao], w,
       yerr=[r["dp_cad"] for r in replicacao], label="Caduceus", color=C_CAD, capsize=3)
ax.bar(xs + w/2, [r["kmers"] for r in replicacao], w,
       yerr=[r["dp_kmer"] for r in replicacao], label="4-mers", color=C_KMER, capsize=3)
vs = [r["caduceus"] for r in replicacao] + [r["kmers"] for r in replicacao]
mg = 0.45 * (max(vs) - min(vs) + 1e-6)
ax.set_ylim(min(vs) - mg, max(vs) + mg); ax.axhline(0, c="black", lw=0.9)
ax.set_xticks(xs); ax.set_xticklabels([r["degrau"] for r in replicacao], fontsize=9)
ax.set_ylabel("Δ AUROC")
ax.set_title("B · Os mesmos degraus, noutra representação", loc="left")
ax.legend(fontsize=8, frameon=False)

fig.text(0.01, -0.15,
         "Cinza: modelo mais simples (média dos embeddings + regressão logística). Preto: modelo completo "
         "(Attention-MIL). Barras azuis sobem, vermelhas descem.\n"
         "NORMALIZAÇÃO troca a referência de padronização; CLASSIFICADOR troca a regressão por uma rede; PROJEÇÃO acrescenta uma camada "
         "DEPOIS da média; ORDEM move essa mesma camada para ANTES;\n"
         "AGREGADOR troca a média por atenção. Como a projeção é afim, ela comuta com a média para quaisquer pesos: as duas "
         "arquiteturas do degrau ORDEM diferem só na posição da ReLU.\n"
         "A decomposição depende do caminho: outra ordem dos degraus daria outras fatias com a mesma soma.",
         fontsize=7.4, style="italic", va="top")
fig.suptitle("Figura 1 — Onde o sinal se perde na cadeia de agregação", fontsize=12,
             fontweight="bold", y=1.02)
fig.savefig(SAIDA / "Figura_1_Escada.png"); fig.savefig(SAIDA / "Figura_1_Escada.pdf")
plt.close(fig); print("  [FIGURA] Figura_1_Escada")

# ============================ FIGURA 2 =======================================
def espalha(v, minimo):
    v = np.asarray(v, float); o = np.argsort(v); s = v.copy()
    for k in range(1, len(o)):
        if s[o[k]] - s[o[k-1]] < minimo: s[o[k]] = s[o[k-1]] + minimo
    return s - (s.mean() - v.mean())

fig, ax = plt.subplots(figsize=(12.0, 9.0))
ks = [inv[n] for n in tab_int["Braco"].tolist()]
yi = np.array([RES[k]["media"] for k in ks]); ye = np.array([RES[k]["auc_ext"] for k in ks])
faixa = max(yi.max(), ye.max()) - min(yi.min(), ye.min())
re_, rd_ = espalha(yi, 0.030*faixa), espalha(ye, 0.030*faixa)
for i, k in enumerate(ks):
    n = nome(k)
    if "Cobertura" in n:    c, lw, al = C_DELFI, 2.4, 1.0
    elif "Caduceus" in n:   c, lw, al = C_CAD, 1.5, 0.85
    elif "4-mers" in n:     c, lw, al = C_KMER, 1.5, 0.85
    elif "End-Motif" in n:  c, lw, al = C_MOTIF, 2.0, 0.95
    else:                   c, lw, al = "#777777", 1.1, 0.6
    ax.plot([0, 1], [yi[i], ye[i]], "-o", color=c, lw=lw, ms=5, alpha=al)
    ax.plot([-0.045, -0.012], [re_[i], yi[i]], "-", color=c, lw=0.5, alpha=0.5)
    ax.text(-0.050, re_[i], CURTO[k], ha="right", va="center", fontsize=7.2, color=c)
    ax.plot([1.012, 1.045], [ye[i], rd_[i]], "-", color=c, lw=0.5, alpha=0.5)
    ax.text(1.050, rd_[i], f"{ye[i]:.3f}", ha="left", va="center", fontsize=7.2, color=c)
todos = np.r_[yi, ye, re_, rd_, 0.5]; mg = 0.05*(todos.max()-todos.min())
ax.set_ylim(todos.min()-mg, todos.max()+mg)
ax.axhline(0.5, ls="--", c="black", lw=1.0)
ax.text(0.5, 0.5+0.10*mg, "acaso", ha="center", fontsize=8, style="italic")
ax.set_xlim(-0.80, 1.36); ax.set_xticks([0, 1])
ax.set_xticklabels([f"Baltimore (n={N_DEV})\npan-câncer vs saudável",
                    f"Hong Kong (n={int(sel.sum())})\ncarcinoma hepatocelular vs saudável"],
                   fontsize=9)
ax.set_ylabel("AUROC")
ax.set_title("Figura 2 — O que sobrevive à mudança de coorte E de tarefa", fontweight="bold")
fig.text(0.01, -0.03,
         "Cada linha liga o desempenho de um modelo nas duas coortes. Verde: coberturas em megabases. "
         "Azul: representações do Caduceus. Vermelho: 4-mers. Laranja: perfil de end-motif. Cinza: controles.\n"
         "ATENÇÃO: isto NÃO é validação externa da mesma tarefa. O alvo externo é um tipo tumoral ausente do treino, numa "
         "população com outra ascendência, centro e protocolo.\n"
         "As três coberturas diferem apenas na região e na profundidade; comparar as três separa esses dois efeitos.",
         fontsize=7.6, style="italic", va="top")
fig.savefig(SAIDA / "Figura_2_Generalizacao.png"); fig.savefig(SAIDA / "Figura_2_Generalizacao.pdf")
plt.close(fig); print("  [FIGURA] Figura_2_Generalizacao")

# ============================ FIGURA 3 =======================================
b_i, b_e = inv[tab_int.iloc[0]["Braco"]], inv[tab_ext.iloc[0]["Braco"]]
if b_i == b_e: b_e = inv[tab_ext.iloc[1]["Braco"]]
papel = {b_i: "maior AUROC no desenvolvimento", b_e: "maior AUROC na coorte externa"}
gr = ["healthy", "hepatitis_b", "cirrhosis", HCC]
bc = {"healthy": "Saudável", "hepatitis_b": "Hepatite B", "cirrhosis": "Cirrose",
      HCC: "Carcinoma\nhepatocelular"}
nc = {g: f"{bc[g]}\n(n={int((cond_ext == g).sum())})" for g in gr}
mk = np.isin(cond_ext, gr)
fig, axes = plt.subplots(1, 2, figsize=(13.0, 6.0))
for ax, b in zip(np.atleast_1d(axes), [b_i, b_e]):
    dd = pd.DataFrame(dict(grupo=[nc[c] for c in cond_ext[mk]], escore=RES[b]["externo"][mk]))
    sns.violinplot(data=dd, x="grupo", y="escore", order=[nc[g] for g in gr], ax=ax,
                   inner="quartile", cut=0, hue="grupo", legend=False,
                   palette=["#BBBBBB", "#F4A582", "#D6604D", "#B2182B"])
    sns.stripplot(data=dd, x="grupo", y="escore", order=[nc[g] for g in gr], ax=ax,
                  color="black", size=2.0, alpha=0.35, jitter=0.22)
    ax.set_title(f"{nome(b)}\n({papel[b]})\nAUROC externa {RES[b]['auc_ext']:.3f}", fontsize=8.5)
    ax.set_xlabel(""); ax.set_ylabel("Probabilidade predita de câncer")
    ax.tick_params(axis="x", labelsize=8.5)
fig.suptitle("Figura 3 — Como os modelos se comportam diante de doença hepática sem tumor",
             fontsize=12, fontweight="bold")
fig.text(0.01, -0.05,
         "Cada ponto é um paciente; as linhas dentro do violino são os quartis. Hepatite B e cirrose são doenças "
         "do fígado SEM tumor e não entram no cálculo da AUROC:\nsua posição mostra se o modelo confunde inflamação com câncer. "
         "Os dois painéis mostram as maiores AUROC observadas, escolhidas depois de ver os resultados.",
         fontsize=7.6, style="italic", va="top")
fig.tight_layout()
fig.savefig(SAIDA / "Figura_3_Grupos_Clinicos.png"); fig.savefig(SAIDA / "Figura_3_Grupos_Clinicos.pdf")
plt.close(fig); print("  [FIGURA] Figura_3_Grupos_Clinicos")

# ============================ FIGURA 4 =======================================
# A dimensão exibida é escolhida em 40 pacientes sorteados e mostrada em outros
# 40, também sorteados, nunca nas primeiras linhas da tabela, que poderiam
# refletir ordem de lote ou de centro em vez de acaso.
rng_f4 = np.random.default_rng(SEMENTE_MESTRA + 999)
pt, nt = IDX_DEV[Y_DEV == 1], IDX_DEV[Y_DEV == 0]
perm_p, perm_n = rng_f4.permutation(pt), rng_f4.permutation(nt)
esc_p, esc_n = np.sort(perm_p[:20]), np.sort(perm_n[:20])
exi_p, exi_n = np.sort(perm_p[20:40]), np.sort(perm_n[20:40])
if len(exi_p) < 5 or len(exi_n) < 5:
    exi_p, exi_n = esc_p, esc_n
dim = int(np.argmax(np.abs(np.asarray(CAD[esc_p], np.float32).mean(axis=(0, 1))
                           - np.asarray(CAD[esc_n], np.float32).mean(axis=(0, 1)))))
Bp = np.asarray(CAD[exi_p], np.float32); Bn = np.asarray(CAD[exi_n], np.float32)
J = np.concatenate([Bp, Bn]); mu, sd = J.mean(axis=(0, 1)), J.std(axis=(0, 1)) + 1e-9
del J
Bp, Bn = (Bp - mu)/sd, (Bn - mu)/sd

fp, fn_ = Bp[:, :, dim].ravel(), Bn[:, :, dim].ravel()
mp, mn  = Bp[:, :, dim].mean(axis=1), Bn[:, :, dim].mean(axis=1)
d_frag_fig = float(np.abs(fp.mean() - fn_.mean()) /
                   (np.sqrt((fp.var(ddof=1) + fn_.var(ddof=1))/2) + 1e-12))
d_pac_fig  = float(np.abs(mp.mean() - mn.mean()) /
                   (np.sqrt((mp.var(ddof=1) + mn.var(ddof=1))/2) + 1e-12))

fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.9))
ax = axes[0]
sns.kdeplot(fn_, ax=ax, fill=True, color=C_DESCE, label="controle", lw=1.4)
sns.kdeplot(fp,  ax=ax, fill=True, color=C_SOBE,  label="caso", lw=1.4)
ax.set_title(f"A · Um fragmento isolado\nd = {d_frag_fig:.4f} (dimensão {dim})", loc="left")
ax.set_xlabel(f"dimensão {dim}, padronizada"); ax.set_ylabel("densidade")
ax.legend(fontsize=8, frameon=False)

ax = axes[1]
sns.kdeplot(mn, ax=ax, fill=True, color=C_DESCE, label="controle", lw=1.4)
sns.kdeplot(mp, ax=ax, fill=True, color=C_SOBE,  label="caso", lw=1.4)
ax.set_title(f"B · Média de {CFG['N_FRAGMENTOS']} fragmentos\n"
             f"d = {d_pac_fig:.3f} (dimensão {dim})", loc="left")
ax.set_xlabel("média por paciente"); ax.set_ylabel("")
ax.legend(fontsize=8, frameon=False)

ax = axes[2]
T = np.concatenate([Bn[:, :, dim], Bp[:, :, dim]])
antes, depois = np.maximum(T, 0).mean(axis=1), np.maximum(T.mean(axis=1), 0)
ax.scatter(depois, antes, s=30, c=C_CAD, alpha=0.7, edgecolors="white", lw=0.4, zorder=3)
lim = [min(depois.min(), antes.min())-0.05, max(depois.max(), antes.max())+0.05]
ax.plot(lim, lim, "--", c="black", lw=1.2, zorder=4); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_title("C · A ordem importa", loc="left")
ax.set_xlabel("ReLU DEPOIS da média"); ax.set_ylabel("ReLU ANTES da média")
ax.text(0.04, 0.95, f"dispersão entre pacientes\nReLU antes:  {np.std(antes):.4f}\n"
        f"ReLU depois: {np.std(depois):.4f}", transform=ax.transAxes, fontsize=8,
        va="top", bbox=dict(boxstyle="round,pad=0.4", fc="#F7F7F7", ec="#CCCCCC"))
fig.suptitle("Figura 4 — Por que a ordem das operações importa", fontsize=12, fontweight="bold")
fig.text(0.01, -0.07,
         f"COMO LER — A e B mostram UMA dimensão, escolhida em 40 pacientes sorteados e exibida em outros 40; os d são os desta "
         f"dimensão, não a mediana das {D_EMB}.\nSobre o desenvolvimento (Célula 1), o erro de estimar a média domina até cerca de "
         f"{N_ESTRELA:.0f} fragmentos de {CFG['N_FRAGMENTOS']} [IC 95% {N_EST_LO:.0f}–{N_EST_HI:.0f}]: os fragmentos de um mesmo "
         f"paciente são correlacionados.\nC: cada ponto é um paciente; se aplicar a ReLU antes ou depois da média fosse indiferente, "
         f"todos cairiam na diagonal. A camada linear comuta com a média,\nentão o desvio da diagonal vem inteiramente da ReLU.",
         fontsize=7.4, style="italic", va="top")
fig.tight_layout()
fig.savefig(SAIDA / "Figura_4_Mecanismo.png"); fig.savefig(SAIDA / "Figura_4_Mecanismo.pdf")
plt.close(fig); print("  [FIGURA] Figura_4_Mecanismo")
del Bp, Bn

# ============================ FIGURA 5 =======================================
# O segundo pilar: quanta profundidade cada representação exige.
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.6))

# painel A — a escada da cobertura, nas duas coortes
ax = axes[0]
braços_cob = ["delfi", "delfi_prom_all", "delfi_prom_matched"]
xs = np.arange(len(braços_cob))
vi = [RES[b]["media"] for b in braços_cob]
ve = [RES[b]["auc_ext"] for b in braços_cob]
ax.plot(xs, vi, "-o", color=C_DELFI, lw=2.2, ms=8, label="Baltimore (interna)")
ax.plot(xs, ve, "--s", color="#7FBF7B", lw=2.2, ms=8, label="Hong Kong (externa)")
for x, a, b in zip(xs, vi, ve):
    ax.text(x, a + 0.012, f"{a:.3f}", ha="center", fontsize=8.5, fontweight="bold",
            color=C_DELFI)
    ax.text(x, b - 0.026, f"{b:.3f}", ha="center", fontsize=8.5, fontweight="bold",
            color="#4A8A4A")
ax.axhline(0.5, ls=":", c="black", lw=1.0)
ax.set_xticks(xs)
ax.set_xticklabels(["painel inteiro\ntodos os fragmentos",
                    "promotora\ntodos os fragmentos",
                    f"promotora\nos mesmos {CFG['N_FRAGMENTOS']:,}"], fontsize=8.5)
ax.set_ylabel("AUROC")
ax.set_title("A · A cobertura perde por profundidade, não por região", loc="left")
ax.legend(fontsize=8, frameon=False, loc="lower left")
for d in degraus_cob_mlp:
    i = 0 if d["etiqueta"] == "REGIÃO" else 1
    ax.annotate(f"{d['etiqueta']}\n{d['delta']:+.3f}",
                xy=(i + 0.5, (vi[i] + vi[i+1]) / 2), ha="center", va="center",
                fontsize=8, fontweight="bold", color="#333333",
                bbox=dict(boxstyle="round,pad=0.3", fc="#FFFFFF", ec="#CCCCCC"))

# painel B — profundidade exigida por representação
ax = axes[1]
itens = [("Caduceus\n(crossover da variância)", forense["Caduceus"]["n_estrela"], C_CAD),
         ("4-mers\n(crossover da variância)", forense["4-mers"]["n_estrela"], C_KMER)]
if PROJECOES:
    alvo = [d for d in PROJECOES if abs(d["meta"] - 0.80) < 1e-9]
    if alvo:
        itens.append(("Perfil de end-motif\n(confiabilidade 0,80)",
                      alvo[0]["fragmentos"], C_MOTIF))
ys = np.arange(len(itens))
ax.barh(ys, [v for _, v, _ in itens], color=[c for _, _, c in itens], height=0.5)
for y, (_, v, _) in zip(ys, itens):
    ax.text(v * 1.10, y, f"{v:,.0f}", va="center", fontsize=9, fontweight="bold")
ax.axvline(CFG["N_FRAGMENTOS"], ls="--", c="black", lw=1.4)
ax.text(CFG["N_FRAGMENTOS"] * 1.06, len(itens) - 0.35,
        f"bolsa disponível\n{CFG['N_FRAGMENTOS']:,}", fontsize=8, style="italic")
ax.set_xscale("log")
ax.set_yticks(ys); ax.set_yticklabels([t for t, _, _ in itens], fontsize=8.5)
ax.set_xlabel("fragmentos por paciente (escala logarítmica)")
ax.set_title("B · Apetite por dados das mesmas moléculas", loc="left")
ax.invert_yaxis()

fig.suptitle("Figura 5 — Profundidade amostral, e não escala genômica",
             fontsize=12, fontweight="bold")
fig.text(0.01, -0.09,
         "A: as três coberturas usam as mesmas janelas de 5 Mb e o mesmo classificador. Do primeiro para o segundo ponto muda a região; "
         "do segundo para o terceiro muda apenas\no número de moléculas contadas. O degrau PROFUNDIDADE é a única comparação de todo o estudo "
         "em que uma coisa só muda e ela é a quantidade de dados.\n"
         "B: as três cifras são estimandos DIFERENTES — crossover da variância para as representações densas, confiabilidade por divisão ao meio "
         "para o perfil de motivos. A ordenação\né qualitativa, não um eixo calibrado. O que ela mostra é que representações discretizadas em "
         "células de contagem exigem profundidade que representações densas não exigem.",
         fontsize=7.4, style="italic", va="top")
fig.tight_layout()
fig.savefig(SAIDA / "Figura_5_Profundidade.png"); fig.savefig(SAIDA / "Figura_5_Profundidade.pdf")
plt.close(fig); print("  [FIGURA] Figura_5_Profundidade")

# ============================ FIGURA 6 =======================================
# Controle positivo da extração: o perfil de end-motif.
fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.0))

ax = axes[0]
top20 = ordem_motif[:20]
cores20 = [C_MOTIF if NOME_MOTIVO[j].startswith("CC") else "#BBBBBB" for j in top20]
ax.bar(np.arange(20), 100 * motif_medio[top20], color=cores20)
ax.axhline(100 / 256, ls="--", c="black", lw=1.2)
ax.text(19.4, 100/256 * 1.25, "esperado ao acaso", ha="right", fontsize=8, style="italic")
ax.set_xticks(np.arange(20))
ax.set_xticklabels([NOME_MOTIVO[j] for j in top20], rotation=90, fontsize=7.5)
ax.set_ylabel("frequência (%)")
ax.set_title(f"A · Os 20 motivos mais frequentes — {100*FRAC_CC:.1f}% começam em CC",
             loc="left")

ax = axes[1]
if CFG["MOTIF_TESTE_METADES"]:
    j_top = int(ordem_motif[0])
    ax.scatter(100 * pa[:, j_top], 100 * pb[:, j_top], s=14, c=C_MOTIF,
               alpha=0.55, edgecolors="white", lw=0.3)
    lo = min((100*pa[:, j_top]).min(), (100*pb[:, j_top]).min())
    hi = max((100*pa[:, j_top]).max(), (100*pb[:, j_top]).max())
    ax.plot([lo, hi], [lo, hi], "--", c="black", lw=1.2)
    ax.set_xlabel(f"primeira metade da bolsa — {NOME_MOTIVO[j_top]} (%)")
    ax.set_ylabel(f"segunda metade — {NOME_MOTIVO[j_top]} (%)")
    ax.set_title(f"B · Concordância entre metades da bolsa\n"
                 f"mediana {CONF_METADE:.3f} · projetada para a bolsa inteira "
                 f"{CONF_BOLSA:.3f}", loc="left")
fig.suptitle("Figura 6 — O perfil de end-motif como controle positivo da extração",
             fontsize=12, fontweight="bold")
fig.text(0.01, -0.10,
         "A: as quatro bases no ponto exato de clivagem, lidas da referência nas coordenadas dos mesmos fragmentos usados em todo o estudo. "
         "Laranja: motivos iniciados em CC,\nassociados na literatura à clivagem por DNASE1L3. O enriquecimento acima do acaso é evidência "
         "independente de que a extração de coordenadas está correta.\n"
         "B: cada ponto é um paciente, com a bolsa dividida ao meio. A dispersão em torno da diagonal é a imprecisão do perfil nesta "
         "profundidade, e é o que impede o perfil de\nservir como representação candidata a classificação.",
         fontsize=7.4, style="italic", va="top")
fig.tight_layout()
fig.savefig(SAIDA / "Figura_6_EndMotif.png"); fig.savefig(SAIDA / "Figura_6_EndMotif.pdf")
plt.close(fig); print("  [FIGURA] Figura_6_EndMotif")

# ---------------------------- exportação -------------------------------------
tab_int.to_csv(SAIDA/"desempenho_interno.csv", index=False)
tab_ext.to_csv(SAIDA/"desempenho_externo.csv", index=False)
tab_op.to_csv(SAIDA/"ponto_de_operacao.csv", index=False)
tab_co.to_csv(SAIDA/"artefato_de_coorte.csv", index=False)
tab_at.to_csv(SAIDA/"diagnostico_atencao.csv", index=False)
tab_clip.to_csv(SAIDA/"controles_otimizacao.csv", index=False)
pd.DataFrame(degraus).to_csv(SAIDA/"escada_principal.csv", index=False)
pd.DataFrame(degraus_cob_mlp).to_csv(SAIDA/"escada_cobertura_rede.csv", index=False)
pd.DataFrame(degraus_cob_log).to_csv(SAIDA/"escada_cobertura_logistica.csv", index=False)
pd.DataFrame(escada_externa).to_csv(SAIDA/"escada_cobertura_externa.csv", index=False)
pd.DataFrame(replicacao).to_csv(SAIDA/"replicacao_degraus.csv", index=False)
pd.DataFrame({nome(k): [r[k]["media"] for r in REPLICAS] for k in CHAVES},
             index=[f"semente_{s}" for s in CFG["SEMENTES"]]).to_csv(
                 SAIDA/"auroc_por_semente.csv")

# perfis de end-motif, para depósito
np.save(SAIDA / "end_motif_features.npy", MOTIF)
np.save(SAIDA / "end_motif_diversity.npy", DIVERSIDADE)
pd.DataFrame(dict(sample_id=META["sample_id"].astype(str).to_numpy(),
                  cortes_usados=USADOS, diversidade=DIVERSIDADE)).to_csv(
    SAIDA / "end_motif_cobertura.csv", index=False)
pd.DataFrame(MOTIF, columns=NOME_MOTIVO).assign(
    sample_id=META["sample_id"].astype(str).to_numpy()).to_csv(
    SAIDA / "end_motif_perfis.csv", index=False)
if PROJECOES:
    pd.DataFrame(PROJECOES).to_csv(SAIDA / "end_motif_profundidade.csv", index=False)

HASH_CODIGO = "ver notebook depositado"
for cand in ["__notebook__.ipynb", "__notebook_source__.ipynb"]:
    p_nb = Path("/kaggle/working") / cand
    if p_nb.exists():
        HASH_CODIGO = hashlib.sha256(p_nb.read_bytes()).hexdigest()[:16]; break

def _num(x):
    """JSON estrito não aceita NaN; devolvemos None quando o valor não existe."""
    if x is None: return None
    v = float(x)
    return None if not np.isfinite(v) else v

json.dump(dict(
    assinatura_configuracao=HASH_CFG, assinatura_codigo=HASH_CODIGO,
    assinatura_dados=HASH_DADOS, arquivos=hashes,
    configuracao={k: str(v) for k, v in CFG.items()}, mapa_condicao=MAPA,
    tarefa_interna="pan-cancer vs saudavel (Baltimore)",
    tarefa_externa="carcinoma hepatocelular vs saudavel (Hong Kong)",
    tarefas_identicas=bool(hcc_dev > 0),
    coorte_externa="EXPLORATORIA_NAO_HOMOLOGA",
    forense_restrita_ao_desenvolvimento=True,
    n_pacientes=int(N_PAC), n_desenvolvimento=int(N_DEV), n_externo=int(len(IDX_EXT)),
    n_bracos=len(CHAVES), n_bracos_neurais=len(NEURAIS),
    # gates
    gate_padding=int(n_zero), gate_vetor_identico={k: _num(v) for k, v in ERROS.items()},
    gate_comutacao_linear_media=_num(ERRO_COMUTA),
    gate_canal_unico=_num(dp_nulo),
    gate_telescopio_principal=_num(discrep),
    gate_telescopio_cobertura_rede=_num(disc_cob_mlp),
    gate_telescopio_cobertura_logistica=_num(disc_cob_log),
    gate_finitude="completa: coberturas, comprimentos; triagem: Caduceus, 4-mers",
    gate_coberturas_distintas=_num(dif_regiao),
    fracao_fora_histograma=_num(frac_fora), esparsidade_ocupacao=_num(frac_zeros),
    # regime
    crossover_variancia=_num(N_ESTRELA),
    crossover_variancia_ic95=[_num(N_EST_LO), _num(N_EST_HI)],
    crossover_por_representacao={k: _num(v["n_estrela"]) for k, v in forense.items()},
    d_cohen_n_por_classe=int(n_d),
    d_cohen_fragmento=_num(D_FRAG), d_cohen_paciente=_num(D_PAC),
    # encoder
    r2_reconstrucao=_num(R2_RESID), r2_so_kmers=_num(R2_KMER),
    r2_so_ocupacao=_num(R2_OCUP), r2_so_ocupacao_padronizada=_num(R2_OCUP_PAD),
    residuo_computavel=bool(RESID_OK), residuo_interpretavel=bool(RESID_INTERPRE),
    nulo_percentil95_residuo=_num(PERC95_RESID),
    nulo_percentil95_caduceus=_num(PERC95_CAD),
    p_empirico_residuo=_num(P_RESID),
    p_empirico_ic_monte_carlo=[_num(p_lo), _num(p_hi)],
    ganho_encoder_sobre_kmers=_num(gan),
    ganho_encoder_ic95=[_num(gan_lo), _num(gan_hi)],
    # otimizacao
    limiar_parada=CFG["LIMIAR_PARADA"],
    inflacao_pico_bruto_sobre_suave=_num(INFLACAO_PICO),
    execucoes_abaixo_do_limiar=N_EXEC_ABAIXO,
    execucoes_totais=N_EXEC_TOTAL,
    falhas_por_lado_do_degrau_ordem=FALHAS_POR_LADO,
    comparacoes_afetadas_pelo_limiar=N_COMP_AFETADAS,
    taxa_maxima_recorte=_num(TAXA_CLIP), avisos_convergencia=int(AVISOS_CONV),
    # escadas
    escada_principal={d["etiqueta"]: dict(
        delta_por_dobra=_num(d["delta"]), dp_sementes=_num(d["dp_sementes"]),
        delta_escore_agrupado=_num(d["boot_delta"]),
        ic95_bootstrap=[_num(d["boot_inf"]), _num(d["boot_sup"])],
        negativos=int(d["neg"]), de=int(d["n_comp"]),
        delta_atingiram=_num(d["delta_ok"]), n_atingiram=int(d["n_ok"]))
        for d in degraus},
    escada_cobertura_rede={d["etiqueta"]: dict(
        delta_por_dobra=_num(d["delta"]), dp_sementes=_num(d["dp_sementes"]),
        delta_escore_agrupado=_num(d["boot_delta"]),
        ic95_bootstrap=[_num(d["boot_inf"]), _num(d["boot_sup"])],
        negativos=int(d["neg"]), de=int(d["n_comp"]))
        for d in degraus_cob_mlp},
    escada_cobertura_logistica={d["etiqueta"]: dict(
        delta_por_dobra=_num(d["delta"]), dp_sementes=_num(d["dp_sementes"]),
        delta_escore_agrupado=_num(d["boot_delta"]),
        ic95_bootstrap=[_num(d["boot_inf"]), _num(d["boot_sup"])],
        negativos=int(d["neg"]), de=int(d["n_comp"]))
        for d in degraus_cob_log},
    escada_cobertura_externa=[{k: (_num(v) if isinstance(v, float) else v)
                               for k, v in d.items()} for d in escada_externa],
    replicacao={r["degrau"]: dict(caduceus=_num(r["caduceus"]), kmers=_num(r["kmers"]),
                kmers_ic95=[_num(r["boot_kmer_inf"]), _num(r["boot_kmer_sup"])])
                for r in replicacao},
    concordancia_entre_replicas={nome(k): _num(RES[k]["concordancia"]) for k in CHAVES},
    concordancia_antes_depois_ordem={
        e: dict(antes=_num(RES[a]["concordancia"]), depois=_num(RES[d_]["concordancia"]))
        for e, a, d_ in PARES_ORDEM},
    # end-motif
    end_motif=dict(
        protocolo_de_origem=PROTOCOLO_ORIGEM,
        ambos_extremos=bool(CFG["MOTIF_AMBOS_EXTREMOS"]),
        cortes_minimo=int(USADOS.min()), cortes_mediana=int(np.median(USADOS)),
        cortes_esperados=int(cortes_esperados),
        perda_maxima_por_borda=_num(perda_borda.max()),
        cortes_por_motivo=_num(MOTIF_POR_CELULA),
        confiabilidade_metades=_num(CONF_METADE),
        confiabilidade_bolsa_inteira=_num(CONF_BOLSA),
        fracao_motivos_CC=_num(FRAC_CC),
        enriquecimento_CC=_num(FRAC_CC / (16/256)),
        motivos_mais_frequentes=TOP_MOTIVOS,
        diversidade_mediana=_num(np.median(DIVERSIDADE)),
        projecoes_de_profundidade=[{k: _num(v) for k, v in d.items()} for d in PROJECOES],
        teto_de_profundidade={k: _num(v) for k, v in TETO.items()},
    ),
    densidade_de_contagem={k: _num(v) for k, v in DENSIDADE.items()},
    dimensao_figura4=int(dim),
    verificacoes_extra=bool(CFG["VERIFICACOES_EXTRA"]),
    tempo_total_segundos=round(time.time()-T_INICIO, 1)),
    open(SAIDA/"procedencia.json", "w"), indent=2, ensure_ascii=False)

# ---------------------------- síntese ----------------------------------------
print("\n" + "=" * 100); print("SÍNTESE"); print("=" * 100)

print(f"\n1. O ENCODER. O vetor do Caduceus é reconstruível a partir da composição de")
print(f"   4-mers com R² = {R2_KMER:.3f}; somar a localização leva a {R2_RESID:.3f}.")
if RESID_OK:
    print(f"   O resíduo discrimina a {RES['rep_cad_resid']['media']:.3f} contra um nulo "
          f"empírico construído")
    print(f"   sobre o próprio resíduo, com o mesmo desenho de réplicas")
    print(f"   (p = {P_RESID:.4f}, IC de Monte Carlo [{p_lo:.4f}, {p_hi:.4f}]).")
    print(f"   Ganho sobre contar 4-mers: {gan:+.3f} de AUROC "
          f"[IC 95% {gan_lo:+.3f}, {gan_hi:+.3f}].")

print(f"\n2. A BOLSA. Um fragmento isolado separa os grupos por d = {D_FRAG:.4f}; a média")
print(f"   por paciente, por d = {D_PAC:.4f} (medianas das {D_EMB} dimensões, "
      f"{n_d} casos e {n_d} controles).")
print(f"   O erro de estimar a média domina até cerca de {N_ESTRELA:.0f} fragmentos de "
      f"{CFG['N_FRAGMENTOS']}")
print(f"   [IC 95% {N_EST_LO:.0f}–{N_EST_HI:.0f}], sob correlação intra-bolsa homogênea.")

print(f"\n3. A ESCADA PRINCIPAL. Maior perda em {pior['etiqueta']}.")
print(f"   Média das diferenças por dobra : {pior['delta']:+.3f} ± {pior['dp_sementes']:.3f} "
      f"(pacientes fixos)")
print(f"   Diferença do escore agrupado   : {pior['boot_delta']:+.3f} "
      f"[IC 95% {pior['boot_inf']:+.3f}, {pior['boot_sup']:+.3f}] (modelos fixos)")
print(f"   Negativo em {pior['neg']} de {pior['n_comp']} comparações.")
if np.isfinite(pior["delta_ok"]) and pior["n_ok"] < pior["n_comp"]:
    print(f"   Restrito às {pior['n_ok']} execuções acima do limiar: {pior['delta_ok']:+.3f} "
          f"({pior['neg_ok']}/{pior['n_ok']} neg).")
print(f"   A projeção comuta com a média para quaisquer pesos (erro {ERRO_COMUTA:.1e}):")
print(f"   as duas arquiteturas diferem só na posição da ReLU.")

n_item = 4
for r in replicacao:
    print(f"\n{n_item}. Degrau {r['degrau']} em 4-mers: {r['kmers']:+.3f} ± {r['dp_kmer']:.3f} "
          f"[IC 95% {r['boot_kmer_inf']:+.3f}, {r['boot_kmer_sup']:+.3f}],")
    print(f"   {r['neg_kmer']}/{r['n_kmer']} negativos, "
          f"{'mesmo sinal' if r['mesmo_sinal'] else 'sinal oposto'} ao Caduceus.")
    n_item += 1

print(f"\n{n_item}. REPRODUTIBILIDADE. Concordância entre réplicas antes e depois do")
print(f"   degrau ORDEM:")
for etiq, antes, depois in PARES_ORDEM:
    print(f"     {etiq:<10s} {RES[antes]['concordancia']:.3f} → "
          f"{RES[depois]['concordancia']:.3f}")
print(f"   Execuções abaixo do limiar: "
      f"{FALHAS_POR_LADO['projeta fragmento a fragmento']} em braços que projetam")
print(f"   fragmento a fragmento, {FALHAS_POR_LADO['agrega antes da não linearidade']} "
      f"em braços que agregam antes da não linearidade.")
n_item += 1

print(f"\n{n_item}. A ESCADA DA COBERTURA. Restringir a região custa "
      f"{_d_reg['delta']:+.3f} de AUROC interna;")
print(f"   restringir a profundidade aos mesmos {CFG['N_FRAGMENTOS']:,} fragmentos custa "
      f"{_d_prof['delta']:+.3f}")
print(f"   [IC 95% {_d_prof['boot_inf']:+.3f}, {_d_prof['boot_sup']:+.3f}], negativo em "
      f"{_d_prof['neg']} de {_d_prof['n_comp']} comparações.")
for d in escada_externa:
    if d["familia"] == "rede":
        print(f"   Na coorte externa, degrau {d['etiqueta']:<14s} {d['delta_externa']:+.3f}")
print(f"   A vantagem da cobertura em megabases é de AMPLITUDE AMOSTRAL, não de")
print(f"   escala genômica. Com profundidade igualada, ela não transfere.")
n_item += 1

print(f"\n{n_item}. CONTROLE POSITIVO DA EXTRAÇÃO. O perfil de end-motif recupera "
      f"{100*FRAC_CC:.1f}% de")
print(f"   motivos iniciados em CC, contra {100*16/256:.1f}% ao acaso "
      f"({FRAC_CC/(16/256):.1f}x). Os mais frequentes:")
print(f"   {', '.join(TOP_MOTIVOS[:5])}.")
print(f"   Confiabilidade da bolsa de {CFG['N_FRAGMENTOS']:,}: {CONF_BOLSA:.3f}. "
      f"É controle de extração,")
print(f"   não representação candidata a classificação.")
n_item += 1

print(f"\n{n_item}. GENERALIZAÇÃO. Maior AUROC interna: {tab_int.iloc[0]['Braco']} "
      f"({tab_int.iloc[0]['AUROC']:.3f}).")
print(f"   Maior AUROC externa: {tab_ext.iloc[0]['Braco']} "
      f"({tab_ext.iloc[0]['AUROC_externa']:.3f}).")
print(f"   O primeiro atinge "
      f"{float(tab_ext.set_index('Braco')['AUROC_externa'].loc[tab_int.iloc[0]['Braco']]):.3f} "
      f"fora. A coorte externa não é homóloga;")
print(f"   ambos os máximos foram escolhidos depois de ver os dados."); n_item += 1

print(f"\n{n_item}. PONTO DE OPERAÇÃO EXPLORATÓRIO. Maior sensibilidade observada:")
print(f"   {tab_op.iloc[0]['Braco']}, especificidade "
      f"{tab_op.iloc[0]['Especificidade']:.3f} e sensibilidade "
      f"{tab_op.iloc[0]['Sensibilidade']:.3f}")
print(f"   em {int(y_ext.sum())} casos e {int((1-y_ext).sum())} controles. "
      f"Não constitui validação clínica.")

print("\n" + "-" * 100); print("ARQUIVOS GERADOS"); print("-" * 100)
for a in sorted(SAIDA.iterdir()):
    print(f"  {a.name:<42s} {a.stat().st_size/1024:8.1f} KB")
print(f"\nTempo total: {(time.time()-T_INICIO)/60:.1f} minutos")
print("=" * 100); print("FIM"); print("=" * 100)

# =============================================================================
# ===== FIM CÉLULA 7 ==========================================================
# =============================================================================

CÉLULA 7 — FIGURAS, EXPORTAÇÃO E SÍNTESE
  [FIGURA] Figura_1_Escada
  [FIGURA] Figura_2_Generalizacao
  [FIGURA] Figura_3_Grupos_Clinicos
  [FIGURA] Figura_4_Mecanismo
  [FIGURA] Figura_5_Profundidade
  [FIGURA] Figura_6_EndMotif

SÍNTESE

1. O ENCODER. O vetor do Caduceus é reconstruível a partir da composição de
   4-mers com R² = 0.990; somar a localização leva a 0.990.
   O resíduo discrimina a 0.604 contra um nulo empírico construído
   sobre o próprio resíduo, com o mesmo desenho de réplicas
   (p = 0.0050, IC de Monte Carlo [0.0000, 0.0183]).
   Ganho sobre contar 4-mers: +0.098 de AUROC [IC 95% +0.054, +0.143].

2. A BOLSA. Um fragmento isolado separa os grupos por d = 0.0104; a média
   por paciente, por d = 0.2031 (medianas das 256 dimensões, 60 casos e 60 controles).
   O erro de estimar a média domina até cerca de 675 fragmentos de 5000
   [IC 95% 509–1012], sob correlação intra-bolsa homogênea.

3. A ESCADA PRINCIPAL. Maior perda em ORDEM.
   Média das diferenças por dobra